# Fig 1E : Hallmark GSEA

In [ ]:
  import gseapy as gp                                                                                                                       
                                                                                                                                            
  gene = 'SLC7A5'                                                                                                                           
                                                                                                                                            
  # ── 1. SLC7A5 pos/neg 레이블 부착 ──────────────────────────────                                                                         
  gene_idx = list(adata.var_names).index(gene)
  expr_vec = adata.X[:, gene_idx]                                                                                                           
  if hasattr(expr_vec, 'toarray'):
      expr_vec = expr_vec.toarray().flatten()                                                                                               
  else:                                                                                                                                     
      expr_vec = np.array(expr_vec).flatten()
                                                                                                                                            
  adata.obs['SLC7A5_group'] = np.where(expr_vec > 0, 'pos', 'neg')

  # ── 2. 3개 celltype 필터 ───────────────────────────────────────                                                                         
  # ※ Hepatocyte가 'celltype' 컬럼에 없으면 아래에서 확인
  # print(adata.obs['celltypist_liver'].unique())                                                                                           
  target_celltypes = ['Hepatocytes', 'Stellete', 'Myeloid','Endothelial','Lymphoid']                                                                                  
  adata_sub = adata[adata.obs['celltype'].isin(target_celltypes)].copy()       

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

deg_outdir = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/SLCposneg_cellseq/")
deg_outdir.mkdir(parents=True, exist_ok=True)

LFC_CAP = 10.0

deg_results = {}
deg_tables = []

for ct in target_celltypes:
    sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()
    grp_counts = sub.obs['SLC7A5_group'].value_counts()

    if 'pos' not in grp_counts or 'neg' not in grp_counts or grp_counts.min() < 3:
        print(f"[{ct}] pos/neg 세포 부족, skip: {grp_counts.to_dict()}")
        continue

    sc.tl.rank_genes_groups(
        sub,
        groupby='SLC7A5_group',
        groups=['pos'],
        reference='neg',
        method='wilcoxon',
        use_raw=False,
        pts=True
    )

    deg = sc.get.rank_genes_groups_df(sub, group='pos').rename(
        columns={'names': 'gene'}
    )

    # ── 디버깅: 처음 한 번만 컬럼 확인 ─────────────────────
    if ct == target_celltypes[0]:
        print(f"[DEBUG] DEG columns: {deg.columns.tolist()}")

    # ── pct 컬럼명 자동 탐지 (scanpy 버전별 차이 대응) ──────
    pos_pct_col = next((c for c in deg.columns 
                        if c.startswith('pct_nz') and ('group' in c or c.endswith('_pos'))), None)
    neg_pct_col = next((c for c in deg.columns 
                        if c.startswith('pct_nz') and ('reference' in c or c.endswith('_neg'))), None)

    # 만약 pct_nz_reference가 없으면 adata에서 직접 계산
    if neg_pct_col is None:
        print(f"[{ct}] pct_nz_reference 컬럼 없음 -> 직접 계산")
        # neg 그룹의 cell mask
        neg_mask = (sub.obs['SLC7A5_group'] == 'neg').values
        X_neg = sub[neg_mask].X
        if hasattr(X_neg, 'toarray'):
            # sparse -> 발현 비율 계산
            pct_neg = np.asarray((X_neg > 0).mean(axis=0)).flatten()
        else:
            pct_neg = (X_neg > 0).mean(axis=0)
        gene_to_pct_neg = dict(zip(sub.var_names, pct_neg))
        deg['pct_nz_reference'] = deg['gene'].map(gene_to_pct_neg)
        neg_pct_col = 'pct_nz_reference'

    if pos_pct_col is None:
        pos_mask = (sub.obs['SLC7A5_group'] == 'pos').values
        X_pos = sub[pos_mask].X
        if hasattr(X_pos, 'toarray'):
            pct_pos = np.asarray((X_pos > 0).mean(axis=0)).flatten()
        else:
            pct_pos = (X_pos > 0).mean(axis=0)
        gene_to_pct_pos = dict(zip(sub.var_names, pct_pos))
        deg['pct_nz_group'] = deg['gene'].map(gene_to_pct_pos)
        pos_pct_col = 'pct_nz_group'

    # ── logFC NaN/inf 처리 ─────────────────────────────
    deg['logfoldchanges'] = pd.to_numeric(deg['logfoldchanges'], errors='coerce')
    deg['logFC_imputed'] = deg['logfoldchanges'].isna() | np.isinf(deg['logfoldchanges'])

    # vectorized 처리 (apply보다 빠르고 명확)
    lfc = deg['logfoldchanges'].values
    p = deg[pos_pct_col].values.astype(float)
    n = deg[neg_pct_col].values.astype(float)

    new_lfc = np.empty_like(lfc, dtype=float)

    for i in range(len(lfc)):
        v = lfc[i]
        if np.isfinite(v):
            new_lfc[i] = np.clip(v, -LFC_CAP, LFC_CAP)
        elif v == np.inf:
            new_lfc[i] = LFC_CAP
        elif v == -np.inf:
            new_lfc[i] = -LFC_CAP
        else:  # NaN
            pi, ni = p[i], n[i]
            if np.isnan(pi) and np.isnan(ni):
                new_lfc[i] = 0.0
            else:
                pi = 0.0 if np.isnan(pi) else pi
                ni = 0.0 if np.isnan(ni) else ni
                if pi > ni:
                    new_lfc[i] = LFC_CAP
                elif ni > pi:
                    new_lfc[i] = -LFC_CAP
                else:
                    new_lfc[i] = 0.0

    deg['logfoldchanges'] = new_lfc

    deg.insert(0, 'celltype', ct)
    deg.insert(1, 'comparison', 'SLC7A5_pos_vs_neg')
    deg.insert(2, 'n_pos', int(grp_counts['pos']))
    deg.insert(3, 'n_neg', int(grp_counts['neg']))

    safe_ct = re.sub(r'[^A-Za-z0-9_.-]+', '_', ct)
    full_path = deg_outdir / f"{safe_ct}_SLC7A5_pos_vs_neg_DEG_all.csv"
    sig_path = deg_outdir / f"{safe_ct}_SLC7A5_pos_vs_neg_DEG_padj0.05_logfc0.25.csv"

    deg.to_csv(full_path, index=False)
    sig = deg[(deg['pvals_adj'] < 0.05) & (deg['logfoldchanges'].abs() > 0.25)].copy()
    sig.to_csv(sig_path, index=False)

    deg_results[ct] = deg
    deg_tables.append(deg)
    print(f"[{ct}] saved: all={len(deg):,}, sig={len(sig):,}, "
          f"imputed_logFC={int(deg['logFC_imputed'].sum())} -> {full_path}")

if deg_tables:
    deg_all = pd.concat(deg_tables, ignore_index=True)
    combined_path = deg_outdir / "All_celltypes_SLC7A5_pos_vs_neg_DEG_all.csv"
    deg_all.to_csv(combined_path, index=False)
    print(f"Combined DEG saved: {combined_path} ({len(deg_all):,} rows)")

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# ── 3. 5개 celltype별 SLC7A5 pos vs neg DEG 저장 ───────────────
deg_outdir = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/SLCposneg_cellseq/")
deg_outdir.mkdir(parents=True, exist_ok=True)

# logFC capping 값 (inf/nan 처리용)
LFC_CAP = 10.0

deg_results = {}
deg_tables = []

for ct in target_celltypes:
    sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()
    grp_counts = sub.obs['SLC7A5_group'].value_counts()

    if 'pos' not in grp_counts or 'neg' not in grp_counts or grp_counts.min() < 3:
        print(f"[{ct}] pos/neg 세포 부족, skip: {grp_counts.to_dict()}")
        continue

    sc.tl.rank_genes_groups(
        sub,
        groupby='SLC7A5_group',
        groups=['pos'],
        reference='neg',
        method='wilcoxon',
        use_raw=False,
        pts=True
    )

    # pts=True 면 pct_nz_group(=pos), pct_nz_reference(=neg) 컬럼이 같이 나옴
    deg = sc.get.rank_genes_groups_df(sub, group='pos').rename(
        columns={'names': 'gene'}
    )

    # ── logFC NaN/inf 처리 ────────────────────────────────
    # 숫자로 강제 변환 (빈칸/문자 -> NaN)
    deg['logfoldchanges'] = pd.to_numeric(deg['logfoldchanges'], errors='coerce')

    # 원본 보존용 플래그 (capping/imputation 됐는지 추적)
    deg['logFC_imputed'] = deg['logfoldchanges'].isna() | np.isinf(deg['logfoldchanges'])

    # pct 정보로 방향 결정해서 ±LFC_CAP 부여
    pos_pct = deg.get('pct_nz_group', pd.Series(np.nan, index=deg.index))
    neg_pct = deg.get('pct_nz_reference', pd.Series(np.nan, index=deg.index))

    # +inf / NaN 이면서 pos에서 더 많이 발현 -> +CAP
    # -inf / NaN 이면서 neg에서 더 많이 발현 -> -CAP
    def _impute_lfc(row):
        v = row['logfoldchanges']
        if np.isfinite(v):
            # 정상값도 cap 범위로 clip
            return float(np.clip(v, -LFC_CAP, LFC_CAP))
        # 비정상값: 부호로 방향 결정
        if v == np.inf:
            return LFC_CAP
        if v == -np.inf:
            return -LFC_CAP
        # NaN인 경우 -> pct 비교로 방향 결정
        p = row.get('pct_nz_group', np.nan)
        n = row.get('pct_nz_reference', np.nan)
        if pd.notna(p) and pd.notna(n):
            return LFC_CAP if p >= n else -LFC_CAP
        return 0.0  # pct 정보도 없으면 0

    deg['logfoldchanges'] = deg.apply(_impute_lfc, axis=1)

    # ── 메타데이터 컬럼 ────────────────────────────────
    deg.insert(0, 'celltype', ct)
    deg.insert(1, 'comparison', 'SLC7A5_pos_vs_neg')
    deg.insert(2, 'n_pos', int(grp_counts['pos']))
    deg.insert(3, 'n_neg', int(grp_counts['neg']))

    # ── 저장 ────────────────────────────────
    safe_ct = re.sub(r'[^A-Za-z0-9_.-]+', '_', ct)
    full_path = deg_outdir / f"{safe_ct}_SLC7A5_pos_vs_neg_DEG_all.csv"
    sig_path = deg_outdir / f"{safe_ct}_SLC7A5_pos_vs_neg_DEG_padj0.05_logfc0.25.csv"

    deg.to_csv(full_path, index=False)

    sig = deg[(deg['pvals_adj'] < 0.05) & (deg['logfoldchanges'].abs() > 0.25)].copy()
    sig.to_csv(sig_path, index=False)

    deg_results[ct] = deg
    deg_tables.append(deg)
    print(f"[{ct}] saved: all={len(deg):,}, sig={len(sig):,}, "
          f"imputed_logFC={int(deg['logFC_imputed'].sum())} -> {full_path}")

if deg_tables:
    deg_all = pd.concat(deg_tables, ignore_index=True)
    combined_path = deg_outdir / "All_celltypes_SLC7A5_pos_vs_neg_DEG_all.csv"
    deg_all.to_csv(combined_path, index=False)
    print(f"Combined DEG saved: {combined_path} ({len(deg_all):,} rows)")

In [ ]:
sc.pl.umap(adata, color=['celltype'])

In [ ]:
  import gseapy as gp                                                                                                                       
                                                                                                                                            
  gene = 'SLC7A5'                                                                                                                           
                                                                                                                                            
  # ── 1. SLC7A5 pos/neg 레이블 부착 ──────────────────────────────                                                                         
  gene_idx = list(adata.var_names).index(gene)
  expr_vec = adata.X[:, gene_idx]                                                                                                           
  if hasattr(expr_vec, 'toarray'):
      expr_vec = expr_vec.toarray().flatten()                                                                                               
  else:                                                                                                                                     
      expr_vec = np.array(expr_vec).flatten()
                                                                                                                                            
  adata.obs['SLC7A5_group'] = np.where(expr_vec > 0, 'pos', 'neg')

  # ── 2. 3개 celltype 필터 ───────────────────────────────────────                                                                         
  # ※ Hepatocyte가 'celltype' 컬럼에 없으면 아래에서 확인
  # print(adata.obs['celltypist_liver'].unique())                                                                                           
  target_celltypes = ['Hepatocytes', 'Stellete', 'Myeloid']                                                                                  
  adata_sub = adata[adata.obs['celltype'].isin(target_celltypes)].copy()       

In [ ]:
for ct in target_celltypes:
      sub = adata_sub[adata_sub.obs['celltype'] == ct]
      counts = sub.obs['SLC7A5_group'].value_counts()                                                                                       
      total = len(sub)                                                                                                                      
      print(f"\n[{ct}] total={total}")                                                                                                      
      for grp in ['pos', 'neg']:                                                                                                            
          n = counts.get(grp, 0)
          print(f"  {grp}: {n} ({n/total*100:.1f}%)") 

In [ ]:
  # ── 3. celltype별 GSEA ─────────────────────────────────────────                                                                         
  gene_sets = 'MSigDB_Hallmark_2020'   # 또는 로컬 .gmt 파일 경로
  results   = {}                                                                                                                            
                  
  for ct in target_celltypes:                                                                                                               
      sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()

      grp_counts = sub.obs['SLC7A5_group'].value_counts()                                                                                   
      if 'pos' not in grp_counts or 'neg' not in grp_counts or grp_counts.min() < 3:
          print(f"[{ct}] pos/neg 세포 부족, skip")                                                                                          
          continue                                                                                                                          
                                                                                                                                            
      # pos vs neg Wilcoxon → ranked gene list                                                                                              
      sc.tl.rank_genes_groups(
          sub,                                                                                                                              
          groupby='SLC7A5_group',
          groups=['pos'],
          reference='neg',
          method='wilcoxon',
          use_raw=False
      )
      deg = sc.get.rank_genes_groups_df(sub, group='pos')
                                                                                                                                            
      ranked = (deg[['names', 'scores']]                                                                                                    
                .dropna()                                                                                                                   
                .drop_duplicates('names')                                                                                                   
                .sort_values('scores', ascending=False)
                .set_index('names')['scores'])

      # prerank GSEA                                                                                                                        
      pre_res = gp.prerank(
          rnk=ranked,                                                                                                                       
          gene_sets=gene_sets,
          threads=4,
          min_size=15,
          max_size=500,
          permutation_num=1000,
          outdir=None,                                                                                                                      
          seed=42,
          verbose=False                                                                                                                     
      )           

      results[ct] = pre_res
      print(f"[{ct}] 완료: {len(pre_res.res2d)} pathways")
                                                                                                                                            
  # ── 4. 결과 요약 출력 ─────────────────────────────────────────
  for ct, pre_res in results.items():                                                                                                       
      print(f"\n=== {ct} | SLC7A5 pos vs neg | top NES ===")
      print(pre_res.res2d.sort_values('NES', ascending=False)                                                                               
                         .head(10)[['Term', 'NES', 'NOM p-val', 'FDR q-val']]) 

In [ ]:
keywords = ['gly', 'fatty acid', 'lipid', 'cycle', 'hif', 'tor','pha']

for ct, pre_res in results.items():
    df = pre_res.res2d.copy()
    
    mask = df['Term'].str.lower().str.contains('|'.join(keywords))
    matched = df[mask].sort_values('NES', ascending=False)
    
    print(f"\n=== {ct} | keyword match ===")
    if matched.empty:
        print("No matching pathways found")
    else:
        print(matched[['Term', 'NES', 'NOM p-val', 'FDR q-val']])

In [ ]:
  # pos, neg 각각 모든 유전자 대비 GSEA
  results_sep = {}  # results_sep[ct]['pos'], results_sep[ct]['neg']                                                                        
                                                                                                                                            
  for ct in target_celltypes:                                                                                                               
      sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()                                                                               
      results_sep[ct] = {}                                                                                                                  
   
      for grp, ref in [('pos', 'neg'), ('neg', 'pos')]:                                                                                     
          grp_cells = sub.obs['SLC7A5_group'] == grp
          if grp_cells.sum() < 3:                                                                                                           
              continue
                                                                                                                                            
          sc.tl.rank_genes_groups(
              sub,
              groupby='SLC7A5_group',                                                                                                       
              groups=[grp],
              reference=ref,                                                                                                                
              method='wilcoxon',
              use_raw=False
          )                                                                                                                                 
          deg = sc.get.rank_genes_groups_df(sub, group=grp)
          ranked = (deg[['names', 'scores']]                                                                                                
                    .dropna()                                                                                                               
                    .drop_duplicates('names')                                                                                               
                    .sort_values('scores', ascending=False)                                                                                 
                    .set_index('names')['scores'])                                                                                          
   
          pre_res = gp.prerank(                                                                                                             
              rnk=ranked,
              gene_sets=gene_sets,                                                                                                          
              threads=4,
              min_size=15,                                                                                                                  
              max_size=500,                                                                                                                 
              permutation_num=1000,
              outdir=None,                                                                                                                  
              seed=42,
              verbose=False                                                                                                                 
          )
          results_sep[ct][grp] = pre_res

In [ ]:
  # 관심 term 키워드 (대소문자 무관하게 매칭)                                                                                               
  keywords = [                                                                                                                              
      'MTOR', 'HIF1', 'AUTOPHAGY',
      'GLYCOLYSIS', 'GLUCONEOGENESIS', 'TCA', 'OXIDATIVE PHOSPHORYLATION',                                                                  
      'Fatty Acid'                                                                                                              
  ]                                                                                                                                         
                                                                                                                                            
  def match_terms(res_df, keywords):                                                                                                        
      matched = []
      for kw in keywords:                                                                                                                   
          hits = res_df[res_df['Term'].str.upper().str.contains(kw)]
          if not hits.empty:                                                                                                                
              matched.append(hits.iloc[0]['Term'])
      return matched     

In [ ]:
  # ── 1. 개별 enrichment plot ───────────────────────────────────                                                                          
  for ct, grp_dict in results_sep.items():                                                                                                  
      for grp, pre_res in grp_dict.items():
          matched = match_terms(pre_res.res2d, keywords)                                                                                    
          print(f"[{ct} | {grp}] matched: {matched}")
                                                                                                                                            
          for term in matched:
              row       = pre_res.res2d[pre_res.res2d['Term'] == term].iloc[0]
              term_data = pre_res.results.get(term, {})                                                                                     
              ranking   = term_data.get('RES', None)
              if ranking is None:                                                                                                           
                  continue
                                                                                                                                            
              hit_idx = (term_data.get('hit_indices')
                         or term_data.get('hits')
                         or term_data.get('hit_index')
                         or [])                                                                                                             
   
              color = '#3498DB' if grp == 'pos' else '#E74C3C'                                                                              
                  
              fig, ax = plt.subplots(figsize=(5, 3))                                                                                        
              ax.plot(ranking, color=color, lw=1.5)
              ax.axhline(0, color='black', lw=0.5, linestyle='--')                                                                          
              ax.axvline(np.argmax(np.abs(ranking)), color='red', lw=0.8, linestyle='--')
              for idx in hit_idx:                                                                                                           
                  ax.axvline(idx, color='grey', lw=0.3, alpha=0.4)
                                                                                                                                            
              nes = row['NES']
              fdr = row['FDR q-val']                                                                                                        
              ax.set_title(f"{ct} | {grp} | {term}\nNES={nes:.2f}, FDR={fdr:.3f}", fontsize=8)
              ax.set_xlabel('Rank', fontsize=7)                                                                                             
              ax.set_ylabel('Enrichment Score', fontsize=7)
              ax.tick_params(labelsize=6)                                                                                                   
              plt.tight_layout()
              #plt.savefig(f'GSEA_{ct}_{term[:30]}.pdf', bbox_inches='tight')
              plt.show()                                                                                                                  

In [ ]:
  def sig_star(p):                                                                                                                          
      if np.isnan(p): return ''                                                                                                             
      if p < 0.001:   return '***'                                                                                                          
      if p < 0.01:    return '**'                                                                                                           
      if p < 0.05:    return '*'
      return ''

  def fmt_pval(p):
      if np.isnan(p): return 'NA'
      return f'{p:.2e}' if p < 0.001 else f'{p:.3f}'

  # all_terms 수집 — ct → grp → pre_res 구조로                                                                                              
  all_terms = set()
  for ct, grp_dict in results_sep.items():                                                                                                  
      for grp, pre_res in grp_dict.items():
          all_terms.update(match_terms(pre_res.res2d, keywords))                                                                            
  all_terms = sorted(all_terms)
                                                                                                                                            
  celltypes_list = list(results_sep.keys())
  linestyles = ['-', '--', ':']
  colors     = {'pos': '#3498DB', 'neg': '#E74C3C'}                                                                                         
   
  n_pw = len(all_terms)                                                                                                                     
  fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5), constrained_layout=True)
  if n_pw == 1:                                                                                                                             
      axes = [axes]
                                                                                                                                            
  for ax, term in zip(axes, all_terms):
      legend_handles = []                                                                                                                   
                  
      for ci, ct in enumerate(celltypes_list):
          ls = linestyles[ci % len(linestyles)]
                                                                                                                                            
          for grp in ['pos', 'neg']:
              if grp not in results_sep[ct]:                                                                                                
                  continue
                                                                                                                                            
              pre_res   = results_sep[ct][grp]
              term_data = pre_res.results.get(term, {})                                                                                     
              es_curve  = term_data.get('RES', None)
              if es_curve is None or len(es_curve) == 0:
                  continue                                                                                                                  
   
              row = pre_res.res2d[pre_res.res2d['Term'] == term]                                                                            
              if row.empty:
                  continue

              nes  = row.iloc[0]['NES']                                                                                                     
              fdr  = row.iloc[0]['FDR q-val']
                                                                                                                                            
              color = colors[grp]
              x     = np.linspace(0, 1, len(es_curve))
              ax.plot(x, es_curve, color=color, lw=2.0, alpha=0.85, ls=ls)                                                                  
   
              star = sig_star(fdr)                                                                                                          
              lbl  = f"{ct} ({grp})  NES={nes:.2f}  FDR={fmt_pval(fdr)}{' '+star if star else ''}"
              legend_handles.append(                                                                                                        
                  plt.Line2D([0], [0], color=color, lw=2, ls=ls, label=lbl)
              )                                                                                                                             
                  
      ax.axhline(0, color='grey', lw=0.8, ls='--')                                                                                          
      ax.set_title(term, fontsize=10, fontweight='bold')
      ax.set_xlabel('Gene rank (normalized)', fontsize=8)                                                                                   
      ax.set_ylabel('Running ES', fontsize=8)
      ax.spines['top'].set_visible(False)                                                                                                   
      ax.spines['right'].set_visible(False)
      ax.legend(handles=legend_handles, fontsize=7, framealpha=0.35,                                                                        
                loc='best', title='line=celltype / color=group', title_fontsize=7)
                                                                                                                                            
  fig.suptitle(   
      'GSEA ES Curves — SLC7A5 pos (blue) vs neg (red) per celltype\n'                                                                      
      '(line style = celltype)',
      fontsize=12, fontweight='bold'                                                                                                        
  )
  #plt.savefig('GSEA_ES_curves_pos_neg.pdf', bbox_inches='tight', dpi=300)                                                                   
  plt.show()  

In [ ]:
  # index를 "Myeloid_pos", "Myeloid_neg" 형태로                                                                                             
  all_terms = set()                                                                                                                         
  for ct, grp_dict in results_sep.items():                                                                                                  
      for grp, pre_res in grp_dict.items():                                                                                                 
          all_terms.update(match_terms(pre_res.res2d, keywords))
  all_terms = sorted(all_terms)
                                                                                                                                            
  row_index = [f"{ct}_{grp}" for ct in results_sep for grp in ['pos', 'neg']
               if grp in results_sep[ct]]                                                                                                   
                  
  nes_mat = pd.DataFrame(index=row_index, columns=all_terms, dtype=float)
  fdr_mat = pd.DataFrame(index=row_index, columns=all_terms, dtype=float)
                                                                                                                                            
  for ct, grp_dict in results_sep.items():
      for grp, pre_res in grp_dict.items():                                                                                                 
          idx = f"{ct}_{grp}"
          for term in all_terms:                                                                                                            
              row = pre_res.res2d[pre_res.res2d['Term'] == term]
              if not row.empty:                                                                                                             
                  nes_mat.loc[idx, term] = row.iloc[0]['NES']
                  fdr_mat.loc[idx, term] = row.iloc[0]['FDR q-val']                                                                         
   
  nes_mat  = nes_mat.astype(float)                                                                                                          
  fdr_mat  = fdr_mat.astype(float)
  sig_size = -np.log10(fdr_mat.fillna(1) + 1e-10)                                                                                           
   
  # pos=파랑, neg=빨강 (행 이름 기준)                                                                                                       
  def row_color(idx_name, nes):
      grp = idx_name.split('_')[-1]                                                                                                         
      return '#3498DB' if grp == 'pos' else '#E74C3C'                                                                                       
                                                                                                                                            
  fig, ax = plt.subplots(figsize=(len(all_terms) * 0.9 + 1, len(row_index) * 0.8 + 1))                                                      
                  
  for i, idx in enumerate(nes_mat.index):                                                                                                   
      for j, term in enumerate(nes_mat.columns):
          nes = nes_mat.loc[idx, term]                                                                                                      
          sz  = max(sig_size.loc[idx, term] * 100, 5)
          fdr = fdr_mat.loc[idx, term]
          if np.isnan(nes):                                                                                                                 
              continue
          color = row_color(idx, nes)                                                                                                       
          ax.scatter(j, i, s=sz, c=color, alpha=0.8, edgecolors='k', linewidths=0.4)
          if not np.isnan(fdr) and fdr < 0.05:                                                                                              
              ax.text(j, i, '*', ha='center', va='center', fontsize=9,
                      fontweight='bold', color='white')                                                                                     
                                                                                                                                            
  ax.set_xticks(range(len(all_terms)))
  ax.set_xticklabels(all_terms, rotation=40, ha='right', fontsize=8)                                                                        
  ax.set_yticks(range(len(row_index)))
  ax.set_yticklabels(nes_mat.index, fontsize=9)                                                                                             
  ax.set_xlim(-0.5, len(all_terms) - 0.5)
  ax.set_ylim(-0.5, len(row_index) - 0.5)                                                                                                   
  ax.grid(True, linestyle='--', alpha=0.3)                                                                                                  
  ax.set_title('SLC7A5 pos (blue) vs neg (red) | GSEA NES\n(size=-log10 FDR, *=FDR<0.05)', fontsize=10)
                                                                                                                                            
  for label, color in [('pos group', '#3498DB'), ('neg group', '#E74C3C')]:                                                                 
      ax.scatter([], [], c=color, s=80, label=label, alpha=0.8)                                                                             
  ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1), fontsize=7)                                                                        
                  
  plt.tight_layout()                                                                                                                        
  #plt.savefig('GSEA_bubble_summary.pdf', bbox_inches='tight')
  plt.show()    

In [ ]:
  import matplotlib.colors as mcolors
  fig, ax = plt.subplots(figsize=(len(all_terms) * 0.9 + 1, len(row_index) * 0.8 + 1))                                                      
                                                                                                                                            
  # NES 범위 기준 colormap 설정
  nes_vals = nes_mat.values.flatten()
  vmax = np.nanmax(np.abs(nes_vals))
  norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
  cmap = plt.cm.RdBu_r  # 양수=빨강, 음수=파랑

  sc = None
  for i, idx in enumerate(nes_mat.index):
      for j, term in enumerate(nes_mat.columns):
          nes = nes_mat.loc[idx, term]
          sz  = max(sig_size.loc[idx, term] * 100, 5)
          fdr = fdr_mat.loc[idx, term]
          if np.isnan(nes):
              continue
          sc = ax.scatter(j, i, s=sz, c=nes, cmap=cmap, norm=norm,
                          alpha=0.9, edgecolors='k', linewidths=0.4)
          if not np.isnan(fdr) and fdr < 0.05:                                                                                              
              ax.text(j, i, '*', ha='center', va='center', fontsize=9,
                      fontweight='bold', color='white')                                                                                     
                  
  # colorbar                                                                                                                                
  if sc is not None:
      cbar = plt.colorbar(sc, ax=ax, fraction=0.02, pad=0.02)                                                                               
      cbar.set_label('NES', fontsize=9)                                                                                                     
   
  # 크기 범례                                                                                                                               
  for sz_val, label in [(50, 'FDR=0.1'), (150, 'FDR=0.01'), (300, 'FDR=0.001')]:
      ax.scatter([], [], c='grey', s=sz_val, alpha=0.7, label=label, edgecolors='k')                                                        
  ax.legend(loc='upper right', bbox_to_anchor=(1.5, 1), fontsize=7,                                                                         
            title='-log10(FDR)', title_fontsize=7)                                                                                          
                                                                                                                                            
  ax.set_xticks(range(len(all_terms)))                                                                                                      
  ax.set_xticklabels(all_terms, rotation=40, ha='right', fontsize=8)
  ax.set_yticks(range(len(row_index)))                                                                                                      
  ax.set_yticklabels(nes_mat.index, fontsize=9)
  ax.set_xlim(-0.5, len(all_terms) - 0.5)                                                                                                   
  ax.set_ylim(-0.5, len(row_index) - 0.5)
  ax.grid(True, linestyle='--', alpha=0.3)                                                                                                  
  ax.set_title('SLC7A5 pos vs neg | GSEA NES\n(color=NES, size=-log10 FDR, *=FDR<0.05)', fontsize=10)                                       
                                                                                                                                            
  plt.tight_layout()                                                                                                                        
  #plt.savefig('GSEA_bubble_summary.pdf', bbox_inches='tight')                                                                               
  plt.show()   

# Fig 1E : KEGG GSEA

In [ ]:
  # KEGG를 이용하기 
  gene = 'SLC7A5'                                                                                                                           
                                                                                                                                            
  # ── 1. SLC7A5 pos/neg 레이블 부착 ──────────────────────────────                                                                         
  gene_idx = list(adata.var_names).index(gene)
  expr_vec = adata.X[:, gene_idx]                                                                                                           
  if hasattr(expr_vec, 'toarray'):
      expr_vec = expr_vec.toarray().flatten()                                                                                               
  else:                                                                                                                                     
      expr_vec = np.array(expr_vec).flatten()
                                                                                                                                            
  adata.obs['SLC7A5_group'] = np.where(expr_vec > 0, 'pos', 'neg')

  # ── 2. 3개 celltype 필터 ───────────────────────────────────────                                                                         
  # ※ Hepatocyte가 'celltype' 컬럼에 없으면 아래에서 확인
  # print(adata.obs['celltypist_liver'].unique())                                                                                           
  target_celltypes = ['Hepatocytes', 'Stellete', 'Myeloid']                                                                                  
  adata_sub = adata[adata.obs['celltype'].isin(target_celltypes)].copy()       

In [ ]:
  import scanpy as sc
  # KEGG용 results_sep 별도 생성
  gene_sets_kegg = 'KEGG_2021_Human'                                                                                                        
  results_sep_kegg = {}                                                                                                                     
   
  for ct in target_celltypes:                                                                                                               
      sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()                                                                             
      results_sep_kegg[ct] = {}                                                                                                             
   
      for grp, ref in [('pos', 'neg'), ('neg', 'pos')]:                                                                                     
          if (sub.obs['SLC7A5_group'] == grp).sum() < 3:                                                                                  
              continue                                                                                                                      
          sc.tl.rank_genes_groups(sub, groupby='SLC7A5_group',                                                                            
                                  groups=[grp], reference=ref,                                                                              
                                  method='wilcoxon', use_raw=False)                                                                       
          deg = sc.get.rank_genes_groups_df(sub, group=grp)                                                                                 
          ranked = (deg[['names', 'scores']].dropna()                                                                                       
                    .drop_duplicates('names')                                                                                               
                    .sort_values('scores', ascending=False)                                                                                 
                    .set_index('names')['scores'])                                                                                          
          pre_res = gp.prerank(rnk=ranked, gene_sets=gene_sets_kegg,                                                                      
                               threads=4, min_size=15, max_size=500,                                                                        
                               permutation_num=1000, outdir=None,                                                                           
                               seed=42, verbose=False)                                                                                      
          results_sep_kegg[ct][grp] = pre_res    

In [ ]:
keywords = ['pi3k','mtor', 'hif', 'auto',
      'glycol', 'tca', 'oxidative',                                                                  
      'fatty' ]

for ct, grp_dict in results_sep_kegg.items():                                                                                             
      for grp, pre_res in grp_dict.items():                                                                                                 
          df = pre_res.res2d.copy()                                                                                                         
          mask = df['Term'].str.lower().str.contains('|'.join(keywords))                                                                    
          matched = df[mask].sort_values('NES', ascending=False)                                                                            
          print(f"\n=== {ct} | {grp} | keyword match ===")
          if matched.empty:                                                                                                                 
              print("No matching pathways found")
          else:                                                                                                                             
              print(matched[['Term', 'NES', 'NOM p-val', 'FDR q-val']])

In [ ]:
  # 관심 term 키워드 (대소문자 무관하게 매칭)                                                                                               
  keywords = ['pi3k','mtor', 'hif', 'auto',
      'glycol', 'tca', 'oxidative',                                                                  
      'fatty' ]                                                                                                                                      
                                                                                                                                            
  def match_terms(res_df, keywords):                                                                                                        
      matched = []
      for kw in keywords:                                                                                                                   
          hits = res_df[res_df['Term'].str.lower().str.contains(kw.lower())]
          if not hits.empty:                                                                                                                
              matched.append(hits.iloc[0]['Term'])
      return matched     

In [ ]:
  # ── 1. 개별 enrichment plot ───────────────────────────────────                                                                          
  for ct, grp_dict in results_sep_kegg.items():                                                                                                  
      for grp, pre_res in grp_dict.items():
          matched = match_terms(pre_res.res2d, keywords)                                                                                    
          print(f"[{ct} | {grp}] matched: {matched}")
                                                                                                                                            
          for term in matched:
              row       = pre_res.res2d[pre_res.res2d['Term'] == term].iloc[0]
              term_data = pre_res.results.get(term, {})                                                                                     
              ranking   = term_data.get('RES', None)
              if ranking is None:                                                                                                           
                  continue
                                                                                                                                            
              hit_idx = (term_data.get('hit_indices')
                         or term_data.get('hits')
                         or term_data.get('hit_index')
                         or [])                                                                                                             
   
              color = '#3498DB' if grp == 'pos' else '#E74C3C'                                                                              
                  
              fig, ax = plt.subplots(figsize=(5, 3))                                                                                        
              ax.plot(ranking, color=color, lw=1.5)
              ax.axhline(0, color='black', lw=0.5, linestyle='--')                                                                          
              ax.axvline(np.argmax(np.abs(ranking)), color='red', lw=0.8, linestyle='--')
              for idx in hit_idx:                                                                                                           
                  ax.axvline(idx, color='grey', lw=0.3, alpha=0.4)
                                                                                                                                            
              nes = row['NES']
              fdr = row['FDR q-val']                                                                                                        
              ax.set_title(f"{ct} | {grp} | {term}\nNES={nes:.2f}, FDR={fdr:.3f}", fontsize=8)
              ax.set_xlabel('Rank', fontsize=7)                                                                                             
              ax.set_ylabel('Enrichment Score', fontsize=7)
              ax.tick_params(labelsize=6)                                                                                                   
              plt.tight_layout()
              #plt.savefig(f'GSEA_{ct}_{term[:30]}.pdf', bbox_inches='tight')
              plt.show()                                                                                                                  

In [ ]:
  def sig_star(p):                                                                                                                          
      if np.isnan(p): return ''                                                                                                             
      if p < 0.001:   return '***'                                                                                                          
      if p < 0.01:    return '**'                                                                                                           
      if p < 0.05:    return '*'
      return ''

  def fmt_pval(p):
      if np.isnan(p): return 'NA'
      return f'{p:.2e}' if p < 0.001 else f'{p:.3f}'

  # all_terms 수집 — ct → grp → pre_res 구조로                                                                                              
  all_terms = set()
  for ct, grp_dict in results_sep_kegg.items():                                                                                                  
      for grp, pre_res in grp_dict.items():
          all_terms.update(match_terms(pre_res.res2d, keywords))                                                                            
  all_terms = sorted(all_terms)
                                                                                                                                            
  celltypes_list = list(results_sep_kegg.keys())
  linestyles = ['-', '--', ':']
  colors     = {'pos': '#3498DB', 'neg': '#E74C3C'}                                                                                         
   
  n_pw = len(all_terms)                                                                                                                     
  fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5), constrained_layout=True)
  if n_pw == 1:                                                                                                                             
      axes = [axes]
                                                                                                                                            
  for ax, term in zip(axes, all_terms):
      legend_handles = []                                                                                                                   
                  
      for ci, ct in enumerate(celltypes_list):
          ls = linestyles[ci % len(linestyles)]
                                                                                                                                            
          for grp in ['pos', 'neg']:
              if grp not in results_sep_kegg[ct]:                                                                                                
                  continue
                                                                                                                                            
              pre_res   = results_sep_kegg[ct][grp]
              term_data = pre_res.results.get(term, {})                                                                                     
              es_curve  = term_data.get('RES', None)
              if es_curve is None or len(es_curve) == 0:
                  continue                                                                                                                  
   
              row = pre_res.res2d[pre_res.res2d['Term'] == term]                                                                            
              if row.empty:
                  continue

              nes  = row.iloc[0]['NES']                                                                                                     
              fdr  = row.iloc[0]['FDR q-val']
                                                                                                                                            
              color = colors[grp]
              x     = np.linspace(0, 1, len(es_curve))
              ax.plot(x, es_curve, color=color, lw=2.0, alpha=0.85, ls=ls)                                                                  
   
              star = sig_star(fdr)                                                                                                          
              lbl  = f"{ct} ({grp})  NES={nes:.2f}  FDR={fmt_pval(fdr)}{' '+star if star else ''}"
              legend_handles.append(                                                                                                        
                  plt.Line2D([0], [0], color=color, lw=2, ls=ls, label=lbl)
              )                                                                                                                             
                  
      ax.axhline(0, color='grey', lw=0.8, ls='--')                                                                                          
      ax.set_title(term, fontsize=10, fontweight='bold')
      ax.set_xlabel('Gene rank (normalized)', fontsize=8)                                                                                   
      ax.set_ylabel('Running ES', fontsize=8)
      ax.spines['top'].set_visible(False)                                                                                                   
      ax.spines['right'].set_visible(False)
      ax.legend(handles=legend_handles, fontsize=7, framealpha=0.35,                                                                        
                loc='best', title='line=celltype / color=group', title_fontsize=7)
                                                                                                                                            
  fig.suptitle(   
      'GSEA ES Curves — SLC7A5 pos (blue) vs neg (red) per celltype\n'                                                                      
      '(line style = celltype)',
      fontsize=12, fontweight='bold'                                                                                                        
  )
  plt.savefig('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Fig1Fv1.pdf', bbox_inches='tight', dpi=300)                                                                   
  plt.show()  

In [ ]:
  def sig_star(p):                                                                                                                          
      if np.isnan(p): return ''                                                                                                             
      if p < 0.001:   return '***'                                                                                                          
      if p < 0.01:    return '**'                                                                                                           
      if p < 0.05:    return '*'
      return ''

  def fmt_pval(p):
      if np.isnan(p): return 'NA'
      return f'{p:.2e}' if p < 0.001 else f'{p:.3f}'

  # all_terms 수집 — ct → grp → pre_res 구조로                                                                                              
  all_terms = set()
  for ct, grp_dict in results_sep_kegg.items():                                                                                                  
      for grp, pre_res in grp_dict.items():
          all_terms.update(match_terms(pre_res.res2d, keywords))                                                                            
  all_terms = sorted(all_terms)
                                                                                                                                            
  celltypes_list = list(results_sep_kegg.keys())
  linestyles = ['-', '--', ':']
  colors     = {'pos': '#3498DB', 'neg': '#E74C3C'}                                                                                         
   
  n_pw = len(all_terms)                                                                                                                     
  fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5), constrained_layout=True)
  if n_pw == 1:                                                                                                                             
      axes = [axes]
                                                                                                                                            
  for ax, term in zip(axes, all_terms):
      legend_handles = []                                                                                                                   
                  
      for ci, ct in enumerate(celltypes_list):
          ls = linestyles[ci % len(linestyles)]
                                                                                                                                            
          for grp in ['pos', 'neg']:
              if grp not in results_sep_kegg[ct]:                                                                                                
                  continue
                                                                                                                                            
              pre_res   = results_sep_kegg[ct][grp]
              term_data = pre_res.results.get(term, {})                                                                                     
              es_curve  = term_data.get('RES', None)
              if es_curve is None or len(es_curve) == 0:
                  continue                                                                                                                  
   
              row = pre_res.res2d[pre_res.res2d['Term'] == term]                                                                            
              if row.empty:
                  continue

              nes  = row.iloc[0]['NES']                                                                                                     
              fdr  = row.iloc[0]['FDR q-val']
                                                                                                                                            
              color = colors[grp]
              x     = np.linspace(0, 1, len(es_curve))
              ax.plot(x, es_curve, color=color, lw=2.0, alpha=0.85, ls=ls)                                                                  
   
              star = sig_star(fdr)                                                                                                          
              lbl  = f"{ct} ({grp})  NES={nes:.2f}  FDR={fmt_pval(fdr)}{' '+star if star else ''}"
              legend_handles.append(                                                                                                        
                  plt.Line2D([0], [0], color=color, lw=2, ls=ls, label=lbl)
              )                                                                                                                             
                  
      ax.axhline(0, color='grey', lw=0.8, ls='--')                                                                                          
      ax.set_title(term, fontsize=10, fontweight='bold')
      ax.set_xlabel('Gene rank (normalized)', fontsize=8)                                                                                   
      ax.set_ylabel('Running ES', fontsize=8)
      ax.spines['top'].set_visible(False)                                                                                                   
      ax.spines['right'].set_visible(False)
      ax.legend(handles=legend_handles, fontsize=7, framealpha=0.35,                                                                        
                loc='best', title='line=celltype / color=group', title_fontsize=7)
                                                                                                                                            
  fig.suptitle(   
      'GSEA ES Curves — SLC7A5 pos (blue) vs neg (red) per celltype\n'                                                                      
      '(line style = celltype)',
      fontsize=12, fontweight='bold'                                                                                                        
  )
  #plt.savefig('GSEA_ES_curves_pos_neg.pdf', bbox_inches='tight', dpi=300)                                                                   
  plt.show()  

In [ ]:
  # index를 "Myeloid_pos", "Myeloid_neg" 형태로                                                                                             
  all_terms = set()                                                                                                                         
  for ct, grp_dict in results_sep_kegg.items():                                                                                                  
      for grp, pre_res in grp_dict.items():                                                                                                 
          all_terms.update(match_terms(pre_res.res2d, keywords))
  all_terms = sorted(all_terms)
                                                                                                                                            
  row_index = [f"{ct}_{grp}" for ct in results_sep_kegg for grp in ['pos', 'neg']
               if grp in results_sep_kegg[ct]]                                                                                                   
                  
  nes_mat = pd.DataFrame(index=row_index, columns=all_terms, dtype=float)
  fdr_mat = pd.DataFrame(index=row_index, columns=all_terms, dtype=float)
                                                                                                                                            
  for ct, grp_dict in results_sep_kegg.items():
      for grp, pre_res in grp_dict.items():                                                                                                 
          idx = f"{ct}_{grp}"
          for term in all_terms:                                                                                                            
              row = pre_res.res2d[pre_res.res2d['Term'] == term]
              if not row.empty:                                                                                                             
                  nes_mat.loc[idx, term] = row.iloc[0]['NES']
                  fdr_mat.loc[idx, term] = row.iloc[0]['FDR q-val']                                                                         
   
  nes_mat  = nes_mat.astype(float)                                                                                                          
  fdr_mat  = fdr_mat.astype(float)
  sig_size = -np.log10(fdr_mat.fillna(1) + 1e-10)                                                                                           
   
  # pos=파랑, neg=빨강 (행 이름 기준)                                                                                                       
  def row_color(idx_name, nes):
      grp = idx_name.split('_')[-1]                                                                                                         
      return '#3498DB' if grp == 'pos' else '#E74C3C'                                                                                       
                                                                                                                                            
  fig, ax = plt.subplots(figsize=(len(all_terms) * 0.9 + 1, len(row_index) * 0.8 + 1))                                                      
                  
  for i, idx in enumerate(nes_mat.index):                                                                                                   
      for j, term in enumerate(nes_mat.columns):
          nes = nes_mat.loc[idx, term]                                                                                                      
          sz  = max(sig_size.loc[idx, term] * 100, 5)
          fdr = fdr_mat.loc[idx, term]
          if np.isnan(nes):                                                                                                                 
              continue
          color = row_color(idx, nes)                                                                                                       
          ax.scatter(j, i, s=sz, c=color, alpha=0.8, edgecolors='k', linewidths=0.4)
          if not np.isnan(fdr) and fdr < 0.05:                                                                                              
              ax.text(j, i, '*', ha='center', va='center', fontsize=9,
                      fontweight='bold', color='white')                                                                                     
                                                                                                                                            
  ax.set_xticks(range(len(all_terms)))
  ax.set_xticklabels(all_terms, rotation=40, ha='right', fontsize=8)                                                                        
  ax.set_yticks(range(len(row_index)))
  ax.set_yticklabels(nes_mat.index, fontsize=9)                                                                                             
  ax.set_xlim(-0.5, len(all_terms) - 0.5)
  ax.set_ylim(-0.5, len(row_index) - 0.5)                                                                                                   
  ax.grid(True, linestyle='--', alpha=0.3)                                                                                                  
  ax.set_title('SLC7A5 pos (blue) vs neg (red) | GSEA NES\n(size=-log10 FDR, *=FDR<0.05)', fontsize=10)
                                                                                                                                            
  for label, color in [('pos group', '#3498DB'), ('neg group', '#E74C3C')]:                                                                 
      ax.scatter([], [], c=color, s=80, label=label, alpha=0.8)                                                                             
  ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1), fontsize=7)                                                                        
                  
  plt.tight_layout()                                                                                                                        
  #plt.savefig('GSEA_bubble_summary.pdf', bbox_inches='tight')
  plt.show()    

In [ ]:
  import matplotlib.colors as mcolors
  fig, ax = plt.subplots(figsize=(len(all_terms) * 0.9 + 1, len(row_index) * 0.8 + 1))                                                      
                                                                                                                                            
  # NES 범위 기준 colormap 설정
  nes_vals = nes_mat.values.flatten()
  vmax = np.nanmax(np.abs(nes_vals))
  norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
  cmap = plt.cm.RdBu_r  # 양수=빨강, 음수=파랑

  sc = None
  for i, idx in enumerate(nes_mat.index):
      for j, term in enumerate(nes_mat.columns):
          nes = nes_mat.loc[idx, term]
          sz  = max(sig_size.loc[idx, term] * 100, 5)
          fdr = fdr_mat.loc[idx, term]
          if np.isnan(nes):
              continue
          sc = ax.scatter(j, i, s=sz, c=nes, cmap=cmap, norm=norm,
                          alpha=0.9, edgecolors='k', linewidths=0.4)
          if not np.isnan(fdr) and fdr < 0.05:                                                                                              
              ax.text(j, i, '*', ha='center', va='center', fontsize=9,
                      fontweight='bold', color='white')                                                                                     
                  
  # colorbar                                                                                                                                
  if sc is not None:
      cbar = plt.colorbar(sc, ax=ax, fraction=0.02, pad=0.02)                                                                               
      cbar.set_label('NES', fontsize=9)                                                                                                     
   
  # 크기 범례                                                                                                                               
  for sz_val, label in [(50, 'FDR=0.1'), (150, 'FDR=0.01'), (300, 'FDR=0.001')]:
      ax.scatter([], [], c='grey', s=sz_val, alpha=0.7, label=label, edgecolors='k')                                                        
  ax.legend(loc='upper right', bbox_to_anchor=(1.5, 1), fontsize=7,                                                                         
            title='-log10(FDR)', title_fontsize=7)                                                                                          
                                                                                                                                            
  ax.set_xticks(range(len(all_terms)))                                                                                                      
  ax.set_xticklabels(all_terms, rotation=40, ha='right', fontsize=8)
  ax.set_yticks(range(len(row_index)))                                                                                                      
  ax.set_yticklabels(nes_mat.index, fontsize=9)
  ax.set_xlim(-0.5, len(all_terms) - 0.5)                                                                                                   
  ax.set_ylim(-0.5, len(row_index) - 0.5)
  ax.grid(True, linestyle='--', alpha=0.3)                                                                                                  
  ax.set_title('SLC7A5 pos vs neg | GSEA NES\n(color=NES, size=-log10 FDR, *=FDR<0.05)', fontsize=10)                                       
                                                                                                                                            
  plt.tight_layout()                                                                                                                        
  #plt.savefig('GSEA_bubble_summary.pdf', bbox_inches='tight')                                                                               
  plt.show()   

# suppl Figure - SLC family

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_monoMAC_detail.h5ad")
adata

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby='celltype',
    method='wilcoxon',
    use_raw=True,
    pts=True
)

sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)
deg_cell = sc.get.rank_genes_groups_df(adata, group=None)
deg_cell

In [ ]:
gene_set = {'SLC1A1','SLC1A2','SLC1A4','SLC1A5','SLC7A2','SLC7A5','SLC7A6','SLC7A7','SLC7A8','SLC7A9','SLC7A11',
         'SLC16A10','SLC25A12', 'SLC25A13','SLC25A18','SLC25A22','SLC25A44','SLC36A1','SLC36A4','SLC38A1',
         'SLC38A2','SLC38A3','SLC38A4','SLC38A6','SLC38A7','SLC38A9','SLC38A10','SLC43A1','SLC43A2'}

genes_in_data = [g for g in gene_set if g in adata.var_names]
print(f"Found {len(genes_in_data)} / {len(gene_set)} genes in adata")

expr = adata[:, genes_in_data].X
if hasattr(expr, 'toarray'):  # sparse matrix 처리
    expr = expr.toarray()

df = pd.DataFrame(expr, index=adata.obs_names, columns=genes_in_data)
df['celltype'] = adata.obs['celltype'].values
mean_expr = df.groupby('celltype')[genes_in_data].mean()

from scipy.stats import zscore
mean_expr_z = mean_expr.apply(zscore, axis=0)

# ── 방법 A: seaborn clustermap (계층적 클러스터링 포함) ──────────────────
sns.clustermap(
    mean_expr_z,
    cmap='RdBu_r',
    center=0,
    figsize=(10, 6),
    linewidths=0.5,
    annot=True,
    fmt='.2f',
    xticklabels=True,
    yticklabels=True,
)

In [ ]:
sc.pl.matrixplot(
    adata,
    var_names=genes_in_data,
    groupby='celltype',
    standard_scale='var',   # 유전자별 0-1 스케일
    cmap='Blues',
    colorbar_title='Scaled\nexpression',
    #save='_gene_set_heatmap.png',
)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Dotplot (ALL clusters from deg_cell) — Mean Expression + %Expressing
# ──────────────────────────────────────────────────────────────────────────

target_genes = ['SLC1A1','SLC1A2','SLC1A4','SLC1A5','SLC7A2','SLC7A5','SLC7A6','SLC7A7','SLC7A8','SLC7A9','SLC7A11',
         'SLC16A10','SLC25A12', 'SLC25A13','SLC25A18','SLC25A22','SLC25A44','SLC36A1','SLC36A4','SLC38A1',
         'SLC38A2','SLC38A3','SLC38A4','SLC38A6','SLC38A7','SLC38A9','SLC38A10','SLC43A1','SLC43A2']
clusters = sorted(deg_cell['group'].unique())

# cluster별 deg table dict화
deg_dict = {
    cl: deg_cell[deg_cell['group'] == cl].set_index('names')
    for cl in clusters
}

def get_raw_expr(adata, gene):
    if adata.raw is None:
        raise ValueError("adata.raw is None")
    if gene not in adata.raw.var_names:
        raise ValueError(f"{gene} not found in raw")
    idx = np.where(adata.raw.var_names == gene)[0][0]
    expr = adata.raw.X[:, idx]
    return expr.toarray().flatten() if sp.issparse(expr) else expr.flatten()
    
# ── 데이터 수집 ────────────────────────────────────────────────────────────
rows_g = []
for gene in target_genes:
    for cl in clusters:
        cl_label = f'Cluster {cl}'
        deg_sub = deg_dict[cl]

        # pct_nz_group
        pct_val = np.nan
        if gene in deg_sub.index and 'pct_nz_group' in deg_sub.columns:
            pct_val = float(deg_sub.loc[gene, 'pct_nz_group']) * 100

        # raw mean expression
        mask = adata.obs['celltype'] == cl
        if gene in adata.raw.var_names:
            expr_vals = get_raw_expr(adata, gene)[mask]
        else:
            expr_vals = np.zeros(mask.sum())
        logfc = np.nan
        if gene in deg_sub.index and 'logfoldchanges' in deg_sub.columns:
            logfc = float(deg_sub.loc[gene, 'logfoldchanges'])

        rows_g.append({
            'gene': gene,
            'cluster': cl,
            'cluster_id': cl,
            'mean_expr': float(expr_vals.mean()),
            'pct': pct_val,
            'logFC': logfc 
        })

df_g = pd.DataFrame(rows_g)
df_g['gene'] = pd.Categorical(df_g['gene'], categories=target_genes, ordered=True)

# ── Figure: Dotplot (axes swapped) ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(len(clusters) * 0.9 + 2, len(target_genes) * 0.9))

cluster_order = [f'{cl}' for cl in clusters]
gene_order = target_genes

cmap = plt.cm.Reds
vmin = 0
vmax = 2

for yi, gene in enumerate(gene_order):           # ← y: gene
    for xi, cl in enumerate(cluster_order):      # ← x: cluster
        row = df_g[(df_g['gene'] == gene) & (df_g['cluster'] == cl)]
        if len(row) == 0:
            continue

        pct_val = float(row['pct'].values[0]) if not pd.isna(row['pct'].values[0]) else 0.0
        #expr_val = float(row['mean_expr'].values[0]
        logfc_val = float(row['logFC'].values[0]) if not pd.isna(row['logFC'].values[0]) else 0.0
        logfc_val = np.clip(logfc_val, 0, 2)

        size = max(4, (pct_val / 100) * 800 * 1.2)
        alpha = 0.85 if pct_val > 0 else 0.2
        #color = cmap((expr_val - vmin) / (vmax - vmin + 1e-9))
        norm_val = (logfc_val - vmin) / (vmax - vmin + 1e-9)
        color = cmap(norm_val)

        ax.scatter(
            xi, yi,
            s=size,
            c=[color],
            edgecolors='grey',
            linewidths=0.5,
            alpha=alpha,
            zorder=3
        )

        ax.text(
            xi + 0.35, yi,                     # ← 핵심 수정
            f'{pct_val:.1f}%',
            ha='left',
            va='center',
            fontsize=8,
            color='#333333'
        )

# 축 설정 (swap 반영)
ax.set_xticks(range(len(cluster_order)))
ax.set_xticklabels(cluster_order, fontsize=10, rotation=45, ha='right')

ax.set_yticks(range(len(gene_order)))
ax.set_yticklabels(gene_order, fontsize=11, style='italic')

ax.set_xlim(-0.6, len(cluster_order) - 0.2)  # 오른쪽 공간 약간 확장
ax.set_ylim(-0.6, len(gene_order) - 0.4)
ax.invert_yaxis()

ax.set_title(
    'Amino acid transporter gene dotplot\n'
    '(Dot size = %Expressing, Dot color = Log2FC)',
    fontsize=11
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# colorbar
sm = plt.cm.ScalarMappable(
    cmap=cmap,
    norm=plt.Normalize(vmin=vmin, vmax=vmax)
)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('log2 Fold Change (0–2)', fontsize=9)

# size legend
import matplotlib.lines as mlines
leg_handles = [
    mlines.Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#888888',
        markeredgecolor='grey',
        markersize=np.sqrt((pct_leg / 100) * 800),
        label=f'{pct_leg}%',
        alpha=0.7
    )
    for pct_leg in [10, 30, 50]
]
ax.grid(False)
ax.legend(
    handles=leg_handles,
    title='% Expressing',
    loc='lower right',
    fontsize=8,
    title_fontsize=8,
    framealpha=0.4
)

plt.tight_layout()
#plt.savefig(f"{FIG_DIR}/all_clusters_genes_dotplot.pdf", dpi=300, bbox_inches='tight')
plt.show()

pt = df_g.pivot_table(
    index='gene',
    columns='cluster',
    values=['mean_expr', 'pct'],
    sort=False
)

pt = pt.reindex(target_genes)

print(pt.round(3).to_string())

# Figure1G continuous correlation

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color=['celltype'])

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt

celltype_col = "celltype" 
gene_of_interest = "SLC7A5"
gene_sets_kegg = "KEGG_2021_Human"
results_sep_kegg = {}

min_cells_per_type = 30
min_nonzero_gene = 10
min_geneset_size = 5
threads = 4

# Use raw/log-normalized expression for detection and pathway scoring.
# Scaled/centered adata.X can contain negative values and should not be used for detection.
use_raw = True
layer = None


In [ ]:
def get_expr_df_from_adata(adata, use_raw=False, layer=None):
    source = "adata.X"
    if use_raw and adata.raw is not None:
        X = adata.raw.X
        var_names = adata.raw.var_names.copy()
        source = "adata.raw.X"
    else:
        if use_raw and adata.raw is None:
            print("Warning: use_raw=True but adata.raw is None; falling back to adata.X or selected layer.")
        X = adata.layers[layer] if layer is not None else adata.X
        var_names = adata.var_names.copy()
        source = f"adata.layers[{layer!r}]" if layer is not None else "adata.X"

    if hasattr(X, "toarray"):
        X = X.toarray()

    expr_df = pd.DataFrame(X.T, index=var_names, columns=adata.obs_names)
    expr_min = np.nanmin(expr_df.values)
    expr_max = np.nanmax(expr_df.values)
    print(f"Expression source: {source}; min={expr_min:.4g}, max={expr_max:.4g}")
    if expr_min < 0:
        print(
            "Warning: expression matrix contains negative values. "
            "Detection rates based on >0 are not true transcript detection; use raw/count/log-normalized expression if available."
        )
    return expr_df


expr_df = get_expr_df_from_adata(adata, use_raw=use_raw, layer=layer)

# Enrichr gene symbols는 대문자 기준이라 var_names를 대문자로 맞춤
expr_df.index = expr_df.index.astype(str).str.upper()
gene_of_interest = gene_of_interest.upper()

# 중복 gene symbol이 생기면 평균으로 collapse
expr_df = expr_df.groupby(expr_df.index).mean()

if gene_of_interest not in expr_df.index:
    raise ValueError(f"{gene_of_interest} not found after uppercasing adata.var_names")


In [ ]:
kegg_dict = gp.get_library(name=gene_sets_kegg, organism="Human")

# adata에 존재하는 gene만 남기기
genes_in_data = set(expr_df.index)
kegg_dict = {
    pathway: [g for g in genes if g in genes_in_data]
    for pathway, genes in kegg_dict.items()
}

# 너무 작은 pathway 제거
kegg_dict = {
    pathway: genes
    for pathway, genes in kegg_dict.items()
    if len(set(genes)) >= min_geneset_size
}

print(f"Loaded pathways: {len(kegg_dict)}")
print(pd.Series({k: len(v) for k, v in kegg_dict.items()}).sort_values(ascending=False).head())

In [ ]:
gene_sets_kegg = "KEGG_2021_Human"

# 1) KEGG 전체 library 불러오기
kegg_all = gp.get_library(name=gene_sets_kegg, organism="Human")

# 2) pathway 이름 먼저 보기
for term in list(kegg_all.keys())[:20]:
    print(term)

# 3) 원하는 term만 직접 선택
wanted_terms = [
    "Glycolysis / Gluconeogenesis",
    "Autophagy - animal",
    "Fatty acid degradation",
    "Citrate cycle (TCA cycle)",
    "Oxidative phosphorylation",
]

# 4) 원하는 term만 gene set dict로 만들기
kegg_selected = {
    term: kegg_all[term]
    for term in wanted_terms
    if term in kegg_all
}

print(kegg_selected.keys())

In [ ]:
all_results = []

for ct, cell_ids in adata.obs.groupby(celltype_col).groups.items():
    cell_ids = list(cell_ids)

    if len(cell_ids) < min_cells_per_type:
        continue

    expr_ct = expr_df[cell_ids].copy()

    # SLC7A5 nonzero cell 수 확인
    n_nonzero = int((expr_ct.loc[gene_of_interest] > 0).sum())
    if n_nonzero < min_nonzero_gene:
        continue

    # ssGSEA
    # 결과는 res2d long-format, 또는 resultsOnSamples 형태로 접근 가능
    ss = gp.ssgsea(
        data=expr_ct,
        gene_sets=kegg_selected,
        outdir=None,
        sample_norm_method="rank",
        threads=threads,
        verbose=True,
    )

    # gseapy 결과를 sample x pathway matrix로 변환
    # res2d columns 예시: Name, Term, ES, NES ...
    # 여기서 Name = sample(cell), Term = pathway
    ss_df = ss.res2d.loc[:, ["Name", "Term", "NES"]].copy()
    ss_mat = ss_df.pivot(index="Name", columns="Term", values="NES")

    # 저장해두고 싶으면
    results_sep_kegg[ct] = ss_mat.copy()

    # SLC7A5 expression 벡터
    x = expr_ct.loc[gene_of_interest, ss_mat.index].values.astype(float)

    for pathway in ss_mat.columns:
        y = ss_mat[pathway].values.astype(float)

        valid = np.isfinite(x) & np.isfinite(y)
        if valid.sum() < min_cells_per_type:
            continue

        x_valid = x[valid]
        y_valid = y[valid]

        if np.std(x_valid) == 0 or np.std(y_valid) == 0:
            rho, pval = np.nan, np.nan
        else:
            stat = spearmanr(x_valid, y_valid, nan_policy="omit")
            rho, pval = stat.statistic, stat.pvalue

        all_results.append(
            {
                "cell_type": ct,
                "pathway": pathway,
                "rho": rho,
                "pval": pval,
                "n_cells": len(cell_ids),
                "n_nonzero_SLC7A5": n_nonzero,
            }
        )

corr_df = pd.DataFrame(all_results)


In [ ]:
corr_df

In [ ]:
corr_df["pval_adj"] = np.nan
valid = corr_df["pval"].notna()
corr_df.loc[valid, "pval_adj"] = multipletests(
    corr_df.loc[valid, "pval"].values,
    method="fdr_bh"
)[1]

def fdr_to_star(p):
    if pd.isna(p):
        return ""
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return ""

corr_df["signif"] = corr_df["pval_adj"].map(fdr_to_star)

# 저장
corr_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/SLC7A5_KEGG2021HUMAN_ssGSEA_spearman_by_celltype.csv", index=False)

In [ ]:
rho_mat = corr_df.pivot(index="pathway", columns="cell_type", values="rho")
star_mat = corr_df.pivot(index="pathway", columns="cell_type", values="signif")

# pathway 정렬
rho_mat = rho_mat.loc[rho_mat.abs().max(axis=1).sort_values(ascending=False).index]
star_mat = star_mat.loc[rho_mat.index, rho_mat.columns]

plt.figure(figsize=(1.2 * rho_mat.shape[1] + 4, 0.35 * rho_mat.shape[0] + 4))
ax = sns.heatmap(
    rho_mat,
    cmap="coolwarm",
    center=0,
    vmin=-0.5,
    vmax=0.5,
    annot=star_mat,
    fmt="",
    linewidths=0.4,
    cbar_kws={"label": "Spearman rho"},
)
ax.set_title("SLC7A5 vs KEGG ssGSEA score")
ax.set_xlabel("Cell type")
ax.set_ylabel("Pathway")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = corr_df.copy()
plot_df["minus_log10_fdr"] = -np.log10(plot_df["pval_adj"].clip(lower=1e-300))

plt.figure(figsize=(1.2 * plot_df["cell_type"].nunique() + 4, 0.35 * plot_df["pathway"].nunique() + 4))
ax = sns.scatterplot(
    data=plot_df,
    x="cell_type",
    y="pathway",
    hue="rho",
    size="minus_log10_fdr",
    palette="coolwarm",
    hue_norm=(-0.5, 0.5),
    sizes=(20, 250),
    edgecolor="black",
)
ax.set_title("SLC7A5–KEGG ssGSEA correlation by cell type")
ax.set_xlabel("Cell type")
ax.set_ylabel("Pathway")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# SLC7A5 detection and dropout-robust pathway scoring

This section checks the cell type-wise SLC7A5 detection distribution and repeats the SLC7A5-pathway correlation using rank-based UCell-like and AUCell-like pathway scores.

In [ ]:
# SLC7A5 detection distribution by cell type
# This assumes expr_df is raw/count/log-normalized expression. If the printed source above has negative values,
# rerun this block after switching to adata.raw or an unscaled count/log-normalized layer.
slc_expr = expr_df.loc[gene_of_interest, adata.obs_names].astype(float)
if slc_expr.min() < 0:
    print(
        "Warning: SLC7A5 expression contains negative values. "
        "The detected column below is not a true detection rate unless expression is unscaled."
    )

slc_det_df = adata.obs[[celltype_col]].copy()
slc_det_df[gene_of_interest] = slc_expr.loc[slc_det_df.index].values
slc_det_df["detected"] = slc_det_df[gene_of_interest] > 0
slc_det_df["expr_for_plot"] = slc_det_df[gene_of_interest]

slc_detection_summary = (
    slc_det_df.groupby(celltype_col, observed=True)
    .agg(
        n_cells=(gene_of_interest, "size"),
        n_detected=("detected", "sum"),
        pct_detected=("detected", lambda x: 100 * x.mean()),
        mean_expr=(gene_of_interest, "mean"),
        median_expr=(gene_of_interest, "median"),
        mean_expr_detected=(gene_of_interest, lambda x: x[x > 0].mean() if (x > 0).any() else np.nan),
        max_expr=(gene_of_interest, "max"),
    )
    .sort_values("pct_detected", ascending=False)
)

out_detection = "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/SLC7A5_detection_by_celltype.csv"
slc_detection_summary.to_csv(out_detection)
slc_detection_summary


In [ ]:
# Plot SLC7A5 detection frequency and expression distribution
order = slc_detection_summary.index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(max(8, 1.2 * len(order)), 4.2))

sns.barplot(
    data=slc_detection_summary.reset_index(),
    x=celltype_col,
    y="pct_detected",
    order=order,
    color="#4C78A8",
    ax=axes[0],
)
axes[0].set_title("SLC7A5 detection by cell type")
axes[0].set_xlabel("Cell type")
axes[0].set_ylabel("Detected cells (%)")
axes[0].tick_params(axis="x", rotation=45)

sns.violinplot(
    data=slc_det_df,
    x=celltype_col,
    y="expr_for_plot",
    order=order,
    inner="quartile",
    cut=0,
    color="#72B7B2",
    ax=axes[1],
)
axes[1].set_title("SLC7A5 expression distribution")
axes[1].set_xlabel("Cell type")
axes[1].set_ylabel("Expression")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(
    "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/SLC7A5_detection_distribution_by_celltype.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.show()


In [ ]:
# Dropout-robust rank-based pathway scores: UCell-like and AUCell-like
# These implementations avoid extra package dependencies and use per-cell gene ranks.
def _filter_gene_sets_for_expr(gene_sets, expr_index, min_size=5):
    expr_genes = set(expr_index)
    filtered = {}
    for term, genes in gene_sets.items():
        genes_in_data = [str(g).upper() for g in genes if str(g).upper() in expr_genes]
        genes_in_data = sorted(set(genes_in_data))
        if len(genes_in_data) >= min_size:
            filtered[term] = genes_in_data
    return filtered


def score_ucell_aucell(expr_ct, gene_sets, max_rank=1500, min_size=5):
    """Return cell x pathway matrices for UCell-like and AUCell-like rank scores."""
    gene_sets = _filter_gene_sets_for_expr(gene_sets, expr_ct.index, min_size=min_size)
    if not gene_sets:
        return pd.DataFrame(index=expr_ct.columns), pd.DataFrame(index=expr_ct.columns)

    max_rank = min(int(max_rank), expr_ct.shape[0])

    # Higher expression gets lower rank number. Ties are averaged, which is conservative for sparse scRNA data.
    ranks = expr_ct.rank(axis=0, ascending=False, method="average", na_option="bottom")

    ucell_scores = {}
    aucell_scores = {}
    for pathway, genes in gene_sets.items():
        r = ranks.loc[genes]
        n = len(genes)

        # UCell-like score: normalized Mann-Whitney/rank-sum score using a max-rank threshold.
        r_clip = r.clip(upper=max_rank + 1)
        min_rank_sum = n * (n + 1) / 2
        denom = n * max_rank - min_rank_sum
        if denom <= 0:
            ucell = pd.Series(np.nan, index=expr_ct.columns)
        else:
            ucell = 1 - ((r_clip.sum(axis=0) - min_rank_sum) / denom)
            ucell = ucell.clip(lower=0, upper=1)

        # AUCell-like score: normalized area under the recovery curve within top max_rank genes.
        aucell = ((max_rank - r + 1).clip(lower=0).sum(axis=0) / (n * max_rank)).clip(lower=0, upper=1)

        ucell_scores[pathway] = ucell
        aucell_scores[pathway] = aucell

    ucell_mat = pd.DataFrame(ucell_scores, index=expr_ct.columns)
    aucell_mat = pd.DataFrame(aucell_scores, index=expr_ct.columns)
    return ucell_mat, aucell_mat


def correlate_slc7a5_with_scores(score_mat, expr_ct, ct, method_name):
    rows = []
    x = expr_ct.loc[gene_of_interest, score_mat.index].values.astype(float)
    n_nonzero = int((expr_ct.loc[gene_of_interest] > 0).sum())

    for pathway in score_mat.columns:
        y = score_mat[pathway].values.astype(float)
        valid = np.isfinite(x) & np.isfinite(y)
        if valid.sum() < min_cells_per_type or np.std(x[valid]) == 0 or np.std(y[valid]) == 0:
            rho, pval = np.nan, np.nan
        else:
            stat = spearmanr(x[valid], y[valid], nan_policy="omit")
            rho, pval = stat.statistic, stat.pvalue

        rows.append(
            {
                "method": method_name,
                "cell_type": ct,
                "pathway": pathway,
                "rho": rho,
                "pval": pval,
                "n_cells": len(score_mat.index),
                "n_nonzero_SLC7A5": n_nonzero,
            }
        )
    return rows

In [ ]:
# Recalculate SLC7A5-pathway correlations with UCell-like and AUCell-like scores
rank_score_results = []
rank_score_mats = {}
rank_gene_sets = _filter_gene_sets_for_expr(kegg_selected, expr_df.index, min_size=min_geneset_size)

for ct, cell_ids in adata.obs.groupby(celltype_col, observed=True).groups.items():
    cell_ids = list(cell_ids)
    if len(cell_ids) < min_cells_per_type:
        continue

    expr_ct = expr_df[cell_ids].copy()
    n_nonzero = int((expr_ct.loc[gene_of_interest] > 0).sum())
    if n_nonzero < min_nonzero_gene:
        continue

    ucell_mat, aucell_mat = score_ucell_aucell(
        expr_ct,
        rank_gene_sets,
        max_rank=1500,
        min_size=min_geneset_size,
    )

    rank_score_mats[(ct, "UCell_like")] = ucell_mat
    rank_score_mats[(ct, "AUCell_like")] = aucell_mat

    rank_score_results.extend(correlate_slc7a5_with_scores(ucell_mat, expr_ct, ct, "UCell_like"))
    rank_score_results.extend(correlate_slc7a5_with_scores(aucell_mat, expr_ct, ct, "AUCell_like"))

rank_corr_df = pd.DataFrame(rank_score_results)
rank_corr_df["pval_adj"] = np.nan
valid = rank_corr_df["pval"].notna()
if valid.any():
    rank_corr_df.loc[valid, "pval_adj"] = multipletests(
        rank_corr_df.loc[valid, "pval"].values,
        method="fdr_bh",
    )[1]
rank_corr_df["signif"] = rank_corr_df["pval_adj"].map(fdr_to_star)

out_rank_corr = "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/SLC7A5_KEGG2021HUMAN_UCell_AUCell_spearman_by_celltype.csv"
rank_corr_df.to_csv(out_rank_corr, index=False)
rank_corr_df

In [ ]:
# Heatmap for UCell-like and AUCell-like correlations
for method_name in ["UCell_like", "AUCell_like"]:
    method_df = rank_corr_df[rank_corr_df["method"] == method_name].copy()
    if method_df.empty:
        continue

    rho_mat_rank = method_df.pivot(index="pathway", columns="cell_type", values="rho")
    star_mat_rank = method_df.pivot(index="pathway", columns="cell_type", values="signif")

    rho_mat_rank = rho_mat_rank.loc[
        rho_mat_rank.abs().max(axis=1).sort_values(ascending=False).index
    ]
    star_mat_rank = star_mat_rank.loc[rho_mat_rank.index, rho_mat_rank.columns]

    plt.figure(figsize=(1.2 * rho_mat_rank.shape[1] + 4, 0.35 * rho_mat_rank.shape[0] + 4))
    ax = sns.heatmap(
        rho_mat_rank,
        cmap="coolwarm",
        center=0,
        vmin=-0.5,
        vmax=0.5,
        annot=star_mat_rank,
        fmt="",
        linewidths=0.4,
        cbar_kws={"label": "Spearman rho"},
    )
    ax.set_title(f"SLC7A5 vs KEGG {method_name} score")
    ax.set_xlabel("Cell type")
    ax.set_ylabel("Pathway")
    plt.tight_layout()
    plt.savefig(
        f"/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/SLC7A5_KEGG_{method_name}_spearman_heatmap.pdf",
        bbox_inches="tight",
        dpi=300,
    )
    plt.show()

In [ ]:
# Compare rho values from ssGSEA, UCell-like, and AUCell-like scoring
ssgsea_compare = corr_df.assign(method="ssGSEA_NES")[["method", "cell_type", "pathway", "rho", "pval_adj", "signif"]]
rank_compare = rank_corr_df[["method", "cell_type", "pathway", "rho", "pval_adj", "signif"]]
method_compare_df = pd.concat([ssgsea_compare, rank_compare], ignore_index=True)

plt.figure(figsize=(1.4 * method_compare_df["cell_type"].nunique() + 4, 0.6 * method_compare_df["pathway"].nunique() + 4))
g = sns.catplot(
    data=method_compare_df,
    x="cell_type",
    y="rho",
    hue="method",
    col="pathway",
    kind="bar",
    col_wrap=2,
    height=3.2,
    aspect=1.4,
    palette=["#4C78A8", "#F58518", "#54A24B"],
    sharey=True,
)
g.set_axis_labels("Cell type", "Spearman rho")
g.set_titles("{col_name}")
for ax in g.axes.flat:
    ax.axhline(0, color="black", linewidth=0.8)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(
    "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/SLC7A5_KEGG_method_comparison_rho.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.show()

method_compare_df

# Figure 1G analysis plan: cross-lineage MASLD metabolic reprogramming

Analysis unit is separated into two levels.

1. **Pseudobulk DEG per lineage**: aggregate cells by `celltype x donor/sample x state`, then compare MASLD vs Healthy within each lineage. This avoids treating cells from the same donor as independent replicates.
2. **Pathway enrichment from lineage-specific ranked genes**: run preranked GSEA for each lineage using the MASLD-vs-Healthy DEG ranking. The main Figure 1G panel should be a **Lineage x Pathway NES heatmap** with FDR annotations.

A supporting panel is generated as a **metabolic gene dotplot** across `lineage x condition`, showing mean expression z-score and percent expressing cells for core mTOR, PI3K-Akt, glycolysis, TCA, fatty-acid degradation, OXPHOS, amino-acid transport, and autophagy/lysosomal genes.


In [ ]:
# Figure 1G: lineage-level MASLD vs Healthy pseudobulk DEG -> pathway NES heatmap
#            plus supporting lineage x condition metabolic gene dotplot.
#
# Outputs:
#   02_Analysis/Figure1G_pseudobulk_metadata.csv
#   02_Analysis/Figure1G_lineage_MASLD_vs_Healthy_DEG.csv
#   02_Analysis/Figure1G_lineage_pathway_GSEA.csv
#   02_Analysis/Figure1G_metabolic_gene_dotplot_values.csv
#   05_Figure/Figure1G_lineage_pathway_NES_heatmap.pdf/.png
#   05_Figure/Figure1G_metabolic_gene_dotplot.pdf/.png

import os
import re
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt

try:
    import gseapy as gp
except ImportError:
    gp = None
    print("gseapy is not installed; DEG and dotplot will run, but GSEA heatmap will be skipped.")

analysis_dir = "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis"
figure_dir = "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure"
os.makedirs(analysis_dir, exist_ok=True)
os.makedirs(figure_dir, exist_ok=True)
import scanpy as sc

# Use the full Figure 1 lineage object. Earlier notebook cells may overwrite `adata`
# with mono/mac detail objects, which cannot support cross-lineage Figure 1G analysis.
figure1g_h5ad_path = "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_Final_cellannot.h5ad"
adata = sc.read(figure1g_h5ad_path)
print(adata)

celltype_col = "celltype"
state_col = "state"
sample_col = "sample"
states = ["Healthy", "MASLD"]

# Normalize labels used across versions of this notebook/object.
adata.obs[state_col] = adata.obs[state_col].astype(str).replace({
    "healthy": "Healthy",
    "Healthy": "Healthy",
    "normal": "Healthy",
    "Normal": "Healthy",
    "control": "Healthy",
    "Control": "Healthy",
    "NAFLD": "MASLD",
    "NASH": "MASLD",
    "MASH": "MASLD",
    "MASLD": "MASLD",
})
adata.obs[celltype_col] = adata.obs[celltype_col].astype(str).replace({
    "Hepatocyte": "Hepatocytes",
    "Stellate": "Stellete",
    "Stellate cells": "Stellete",
    "Stellete": "Stellete",
    "Myeloid cells": "Myeloid",
    "B cells": "B cell",
    "Cholangiocytes": "Cholangiocyte",
})

requested_lineages = ["Hepatocytes", "Stellete", "Myeloid", "Endothelial", "Lymphoid", "B cell", "Plasma cells", "Cholangiocyte"]
present_lineages = adata.obs[celltype_col].astype(str).value_counts()
lineage_order = [ct for ct in requested_lineages if ct in present_lineages.index]
print("Figure 1G lineages present:", lineage_order)
print("State counts:")
display(adata.obs[state_col].value_counts())

min_cells_per_pseudobulk = 20
min_replicates_per_state = 2
min_gsea_genes = 100

# Prefer raw/count-like expression for pseudobulk and dotplot. If adata.raw is not available, use adata.X.
expr_adata = adata.raw.to_adata() if getattr(adata, "raw", None) is not None else adata
var_names = pd.Index([str(g).upper() for g in expr_adata.var_names])

# Keep first occurrence when gene symbols are duplicated.
first_gene_pos = {}
for i, g in enumerate(var_names):
    first_gene_pos.setdefault(g, i)

def _clean_sample_id(sample, state):
    sample = str(sample)
    state = str(state)
    # Collapse Healthy CD45 fraction libraries into donor-level replicates.
    m = re.match(r"^(Healthy\d+)", sample)
    if m:
        donor_num = re.sub(r"\D", "", m.group(1))
        return f"Healthy_{donor_num}"
    m = re.match(r"^Healthy[_-]?(\d+)", sample)
    if m:
        return f"Healthy_{m.group(1)}"
    return sample if sample and sample != "nan" else state

meta = adata.obs[[celltype_col, state_col, sample_col]].copy()
meta[celltype_col] = meta[celltype_col].astype(str)
meta[state_col] = meta[state_col].astype(str)
meta[sample_col] = meta[sample_col].astype(str)
meta["pseudobulk_sample"] = [
    _clean_sample_id(s, st) for s, st in zip(meta[sample_col], meta[state_col])
]
meta = meta[meta[celltype_col].isin(lineage_order) & meta[state_col].isin(states)].copy()

X = expr_adata.X
if not sp.issparse(X):
    X = np.asarray(X)

grouped = meta.groupby([celltype_col, state_col, "pseudobulk_sample"], observed=True)
group_sizes = grouped.size().rename("n_cells").reset_index()
print("Candidate pseudobulk groups by lineage/state:")
display(group_sizes.groupby([celltype_col, state_col], observed=True)["n_cells"].agg(["count", "min", "median", "max"]).fillna(0))

effective_min_cells = int(min_cells_per_pseudobulk)
if group_sizes.empty:
    raise ValueError(
        "No candidate pseudobulk groups were found after filtering to lineage_order and Healthy/MASLD. "
        "Check `celltype_col`, `state_col`, and `lineage_order`."
    )
if (group_sizes["n_cells"] >= effective_min_cells).sum() == 0:
    max_group_size = int(group_sizes["n_cells"].max())
    effective_min_cells = max(1, min(5, max_group_size))
    print(
        f"No pseudobulk group passed min_cells_per_pseudobulk={min_cells_per_pseudobulk}. "
        f"Using effective_min_cells={effective_min_cells} for this sparse object. "
        "For the final figure, prefer rerunning on the full object or increasing cells per sample if available."
    )

pb_rows = []
pb_mats = []
for (ct, st, sample_id), idx_labels in grouped.groups.items():
    idx = adata.obs_names.get_indexer(pd.Index(idx_labels).astype(str))
    if (idx < 0).any():
        idx = adata.obs_names.get_indexer(idx_labels)
    idx = idx[idx >= 0]
    if len(idx) < effective_min_cells:
        continue
    block = X[idx, :]
    summed = np.asarray(block.sum(axis=0)).ravel() if sp.issparse(block) else block.sum(axis=0)
    pb_mats.append(summed)
    pb_rows.append({
        "cell_type": ct,
        "state": st,
        "sample": sample_id,
        "n_cells": int(len(idx)),
        "min_cells_cutoff_used": effective_min_cells,
    })

pb_meta = pd.DataFrame(pb_rows)
pb_counts = pd.DataFrame(np.vstack(pb_mats), index=pb_meta.index, columns=var_names) if pb_mats else pd.DataFrame()
pb_meta.to_csv(f"{analysis_dir}/Figure1G_pseudobulk_metadata.csv", index=False)

if pb_counts.empty:
    preview = group_sizes.sort_values("n_cells", ascending=False).head(20)
    display(preview)
    raise ValueError(
        "No pseudobulk profiles were created even after adaptive cutoff. "
        "Inspect the displayed candidate groups and check whether the active `adata` contains the expected Figure 1 lineages."
    )

# If the selected matrix has negative values, it is likely scaled expression. In that case use pseudobulk means directly.
has_negative = np.nanmin(pb_counts.values) < 0
if has_negative:
    print("Warning: expression contains negative values; using pseudobulk mean expression instead of CPM normalization.")
    pb_expr = pb_counts.copy()
else:
    libsize = pb_counts.sum(axis=1).replace(0, np.nan)
    pb_expr = np.log2(pb_counts.div(libsize, axis=0) * 1e6 + 1)

# Pseudobulk MASLD-vs-Healthy DEG per lineage.
deg_rows = []
for ct in lineage_order:
    ct_idx = pb_meta.index[pb_meta["cell_type"] == ct]
    ct_meta = pb_meta.loc[ct_idx]
    n_by_state = ct_meta["state"].value_counts()
    if any(n_by_state.get(st, 0) < min_replicates_per_state for st in states):
        print(f"[{ct}] skipped DEG/GSEA: pseudobulk replicates per state = {dict(n_by_state)}")
        continue

    h_idx = ct_meta.index[ct_meta["state"] == "Healthy"]
    m_idx = ct_meta.index[ct_meta["state"] == "MASLD"]
    h = pb_expr.loc[h_idx]
    m = pb_expr.loc[m_idx]

    mean_h = h.mean(axis=0)
    mean_m = m.mean(axis=0)
    log2fc = mean_m - mean_h
    stat, pval = ttest_ind(m.values, h.values, axis=0, equal_var=False, nan_policy="omit")
    pval = pd.Series(pval, index=pb_expr.columns).replace([np.inf, -np.inf], np.nan)
    padj = pd.Series(np.nan, index=pval.index)
    ok = pval.notna()
    if ok.any():
        padj.loc[ok] = multipletests(pval.loc[ok].clip(lower=1e-300), method="fdr_bh")[1]

    rank_metric = np.sign(log2fc) * -np.log10(pval.clip(lower=1e-300))
    rank_metric = rank_metric.replace([np.inf, -np.inf], np.nan).fillna(log2fc)

    tmp = pd.DataFrame({
        "cell_type": ct,
        "gene": pb_expr.columns,
        "mean_logCPM_MASLD": mean_m.values,
        "mean_logCPM_Healthy": mean_h.values,
        "log2FC_MASLD_vs_Healthy": log2fc.values,
        "t_stat": pd.Series(stat, index=pb_expr.columns).values,
        "pval": pval.values,
        "padj": padj.values,
        "rank_metric": rank_metric.values,
        "n_pb_MASLD": len(m_idx),
        "n_pb_Healthy": len(h_idx),
    })
    deg_rows.append(tmp)

deg_df = pd.concat(deg_rows, ignore_index=True) if deg_rows else pd.DataFrame()
deg_df.to_csv(f"{analysis_dir}/Figure1G_lineage_MASLD_vs_Healthy_DEG.csv", index=False)

# Curated pathway display order for the main heatmap.
pathway_patterns = {
    "mTOR signaling": ["mtor signaling", "mtorc1 signaling"],
    "PI3K-Akt/mTOR": ["pi3k-akt", "pi3k akt", "pi3k_akt_mtor"],
    "Autophagy": ["autophagy"],
    "Glycolysis": ["glycolysis"],
    "TCA cycle": ["citrate cycle", "tca cycle"],
    "Fatty acid degradation": ["fatty acid degradation", "fatty acid metabolism"],
    "Oxidative phosphorylation": ["oxidative phosphorylation"],
    "BCAA degradation": ["valine leucine and isoleucine degradation", "valine leucine isoleucine degradation"],
    "Pyruvate metabolism": ["pyruvate metabolism"],
    "Propanoate metabolism": ["propanoate metabolism"],
    "Linoleic acid metabolism": ["linoleic acid metabolism"],
}

def _norm_term(x):
    return re.sub(r"[^a-z0-9]+", " ", str(x).lower()).strip()

def _select_gene_sets():
    if gp is None:
        return {}
    selected = {}
    libraries = ["KEGG_2021_Human", "MSigDB_Hallmark_2020"]
    for lib in libraries:
        try:
            lib_sets = gp.get_library(name=lib, organism="Human")
        except Exception as e:
            print(f"Could not load {lib}: {e}")
            continue
        for term, genes in lib_sets.items():
            nt = _norm_term(term)
            for display, patterns in pathway_patterns.items():
                if any(_norm_term(pat) in nt for pat in patterns):
                    key = display
                    genes_upper = sorted({str(g).upper() for g in genes if str(g).upper() in first_gene_pos})
                    if len(genes_upper) >= 5:
                        # Prefer the larger overlap if KEGG and Hallmark both map to the same display label.
                        if key not in selected or len(genes_upper) > len(selected[key]):
                            selected[key] = genes_upper
    return selected

selected_gene_sets = _select_gene_sets()
print("Selected gene sets:", {k: len(v) for k, v in selected_gene_sets.items()})

# Preranked GSEA per lineage.
gsea_rows = []
if gp is not None and selected_gene_sets and not deg_df.empty:
    for ct in lineage_order:
        sub = deg_df[deg_df["cell_type"] == ct].copy()
        sub = sub.replace([np.inf, -np.inf], np.nan).dropna(subset=["gene", "rank_metric"])
        sub = sub.drop_duplicates("gene").sort_values("rank_metric", ascending=False)
        if sub.shape[0] < min_gsea_genes:
            print(f"[{ct}] skipped GSEA: only {sub.shape[0]} ranked genes")
            continue
        ranked = sub.set_index("gene")["rank_metric"]
        try:
            pre = gp.prerank(
                rnk=ranked,
                gene_sets=selected_gene_sets,
                outdir=None,
                min_size=5,
                max_size=500,
                permutation_num=1000,
                seed=7,
                threads=4,
                verbose=False,
            )
            res = pre.res2d.copy()
            term_col = "Term" if "Term" in res.columns else "term"
            nes_col = "NES" if "NES" in res.columns else "nes"
            fdr_col = "FDR q-val" if "FDR q-val" in res.columns else "fdr"
            for _, r in res.iterrows():
                gsea_rows.append({
                    "cell_type": ct,
                    "pathway": r[term_col],
                    "NES": float(r[nes_col]),
                    "FDR": float(r[fdr_col]),
                })
        except Exception as e:
            print(f"[{ct}] GSEA failed: {e}")

gsea_df = pd.DataFrame(gsea_rows)
gsea_df.to_csv(f"{analysis_dir}/Figure1G_lineage_pathway_GSEA.csv", index=False)

if not gsea_df.empty:
    pathway_order = [p for p in pathway_patterns if p in gsea_df["pathway"].unique()]
    nes_mat = gsea_df.pivot(index="pathway", columns="cell_type", values="NES").reindex(index=pathway_order, columns=lineage_order)
    fdr_mat = gsea_df.pivot(index="pathway", columns="cell_type", values="FDR").reindex(index=pathway_order, columns=lineage_order)
    annot = fdr_mat.applymap(lambda x: "*" if pd.notna(x) and x < 0.05 else "")

    plt.figure(figsize=(max(7, 1.05 * len(lineage_order) + 2.5), max(4.5, 0.42 * len(pathway_order) + 2)))
    ax = sns.heatmap(
        nes_mat,
        cmap="RdBu_r",
        center=0,
        vmin=-2.5,
        vmax=2.5,
        linewidths=0.5,
        linecolor="white",
        annot=annot,
        fmt="",
        cbar_kws={"label": "NES (MASLD vs Healthy)"},
    )
    ax.set_title("Figure 1G main: lineage-level pathway enrichment")
    ax.set_xlabel("Lineage")
    ax.set_ylabel("Pathway")
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    plt.savefig(f"{figure_dir}/Figure1G_lineage_pathway_NES_heatmap.pdf", bbox_inches="tight", dpi=300)
    plt.savefig(f"{figure_dir}/Figure1G_lineage_pathway_NES_heatmap.png", bbox_inches="tight", dpi=300)
    plt.show()
else:
    print("No GSEA results were produced. Check gseapy/library access and replicate counts.")

# Supporting panel: core metabolic gene dotplot across lineage x condition.
metabolic_genes = [
    "MTOR", "RPS6KB1", "EIF4EBP1", "RHEB", "RPTOR",
    "PIK3CA", "PIK3CD", "AKT1", "AKT2",
    "HK2", "PFKFB3", "PKM", "LDHA", "SLC2A1",
    "CS", "IDH1", "OGDH", "SDHA", "FH",
    "CPT1A", "ACADM", "ACADVL", "HADHA", "ECHS1",
    "NDUFA1", "NDUFB2", "COX4I1", "ATP5F1A",
    "SLC7A5", "SLC3A2", "SLC1A5", "SLC38A1",
    "LAMTOR1", "LAMTOR2", "LAMTOR3", "LAMTOR4", "LAMTOR5", "RRAGA", "RRAGC", "SQSTM1",
]
metabolic_genes = [g for g in metabolic_genes if g in first_gene_pos]

plot_rows = []
for ct in lineage_order:
    for st in states:
        idx = adata.obs_names.get_indexer(meta.index[(meta[celltype_col] == ct) & (meta[state_col] == st)])
        idx = idx[idx >= 0]
        if len(idx) == 0:
            continue
        for gene in metabolic_genes:
            pos = first_gene_pos[gene]
            vals = X[idx, pos]
            vals = np.asarray(vals.toarray()).ravel() if sp.issparse(vals) else np.asarray(vals).ravel()
            plot_rows.append({
                "cell_type": ct,
                "state": st,
                "group": f"{ct}-{st}",
                "gene": gene,
                "mean_expr": float(np.nanmean(vals)),
                "pct_expr": float(100 * np.nanmean(vals > 0)),
                "n_cells": int(len(idx)),
            })

dot_df = pd.DataFrame(plot_rows)
if not dot_df.empty:
    dot_df["mean_expr_z"] = dot_df.groupby("gene")["mean_expr"].transform(
        lambda s: (s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) > 0 else 1)
    )
    group_order = [f"{ct}-{st}" for ct in lineage_order for st in states if f"{ct}-{st}" in dot_df["group"].unique()]
    gene_order = [g for g in metabolic_genes if g in dot_df["gene"].unique()]
    dot_df["group"] = pd.Categorical(dot_df["group"], categories=group_order, ordered=True)
    dot_df["gene"] = pd.Categorical(dot_df["gene"], categories=gene_order[::-1], ordered=True)
    dot_df.to_csv(f"{analysis_dir}/Figure1G_metabolic_gene_dotplot_values.csv", index=False)

    plot_df = dot_df.dropna(subset=["group", "gene"]).copy()
    x_codes = plot_df["group"].cat.codes
    y_codes = plot_df["gene"].cat.codes
    sizes = np.clip(plot_df["pct_expr"], 0, 100) * 3.0 + 5

    plt.figure(figsize=(max(9, 0.45 * len(group_order) + 4), max(7, 0.22 * len(gene_order) + 3)))
    sc_plot = plt.scatter(
        x_codes,
        y_codes,
        s=sizes,
        c=plot_df["mean_expr_z"],
        cmap="RdBu_r",
        vmin=-2,
        vmax=2,
        edgecolor="0.75",
        linewidth=0.35,
    )
    plt.xticks(range(len(group_order)), group_order, rotation=60, ha="right")
    plt.yticks(range(len(gene_order)), gene_order[::-1])
    plt.xlabel("Lineage-condition")
    plt.ylabel("Gene")
    cbar = plt.colorbar(sc_plot)
    cbar.set_label("Mean expression z-score")

    for pct, xpos in [(25, len(group_order) + 0.4), (50, len(group_order) + 1.0), (75, len(group_order) + 1.6)]:
        plt.scatter([xpos], [-1.1], s=pct * 3.0 + 5, color="white", edgecolor="0.4", linewidth=0.5)
        plt.text(xpos, -2.0, f"{pct}%", ha="center", va="top", fontsize=8)
    plt.text(len(group_order) + 1.0, -3.2, "% expressing", ha="center", va="top", fontsize=8)

    plt.title("Figure 1G support: metabolic genes by lineage and condition")
    plt.grid(axis="both", color="0.9", linewidth=0.5)
    plt.tight_layout()
    plt.savefig(f"{figure_dir}/Figure1G_metabolic_gene_dotplot.pdf", bbox_inches="tight", dpi=300)
    plt.savefig(f"{figure_dir}/Figure1G_metabolic_gene_dotplot.png", bbox_inches="tight", dpi=300)
    plt.show()
else:
    print("No dotplot data were produced. Check gene symbols and lineage/state labels.")

display(pb_meta.groupby(["cell_type", "state"], observed=True)["sample"].nunique().unstack(fill_value=0))
display(deg_df.head())
display(gsea_df.head() if not gsea_df.empty else pd.DataFrame())


# Fig 1E (개선): SLC7A5 Top vs Bottom Percentile Hallmark GSEA

기존 pos/neg (발현 유무) 방식의 문제점:
- Hepatocytes: pos=6개(4.1%), Stellate: pos=29개(1.8%) — 극단적 불균형
- Duplicated Wilcoxon score 90%+로 ranking 신뢰도 낮음

개선: **celltype별 세포 수에 따른 적응형 percentile**
- ≥1000개 → top/bottom 10%
- 500–999개 → top/bottom 15%
- <500개 → top/bottom 20%
- 중간 구간 제외 → 신호 명확화, 그룹 균형 확보

In [ ]:
# ── Fig 1E 개선: SLC7A5 Top/Bottom Percentile Hallmark GSEA ────────────────
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# Cell 94가 adata를 덮어쓰므로 분석용 adata 별도 로드
_MONO_H5AD = "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_monoMAC_detail.h5ad"
adata_mono = sc.read(_MONO_H5AD)

GENE = "SLC7A5"
GENE_SETS = "MSigDB_Hallmark_2020"
LFC_CAP = 10.0
MIN_CELLS_PER_GROUP = 25

TARGET_CELLTYPES = ["Hepatocytes", "Stellete", "Myeloid", "Endothelial", "Lymphoid"]
OUTDIR_FIG = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure")
OUTDIR_ANA = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/SLC7A5_topbottom_pct")
OUTDIR_ANA.mkdir(parents=True, exist_ok=True)

# ── 1. SLC7A5 발현 추출 ────────────────────────────────────────
adata_sub2 = adata_mono[adata_mono.obs["celltype"].isin(TARGET_CELLTYPES)].copy()

gene_idx = list(adata_sub2.var_names).index(GENE)
expr_vec = adata_sub2.X[:, gene_idx]
if hasattr(expr_vec, "toarray"):
    expr_vec = expr_vec.toarray().flatten()
else:
    expr_vec = np.asarray(expr_vec).flatten()
adata_sub2.obs["SLC7A5_expr"] = expr_vec

# ── 2. celltype별 적응형 percentile ────────────────────────────
def adaptive_pct(n_cells):
    if n_cells >= 1000:
        return 10
    elif n_cells >= 500:
        return 15
    else:
        return 20

print(f"{'CellType':<15} {'Total':>6} {'Pct':>5} {'N/group':>8}")
print("-" * 38)
for ct in TARGET_CELLTYPES:
    n = int((adata_sub2.obs["celltype"] == ct).sum())
    pct = adaptive_pct(n)
    n_g = int(n * pct / 100)
    flag = "" if n_g >= MIN_CELLS_PER_GROUP else "  <- skip"
    print(f"{ct:<15} {n:>6} {pct:>4}% {n_g:>8}{flag}")

# ── 3. celltype별 DEG -> prerank GSEA ───────────────────────────
results_pct = {}

for ct in TARGET_CELLTYPES:
    sub = adata_sub2[adata_sub2.obs["celltype"] == ct].copy()
    n = len(sub)
    pct = adaptive_pct(n)
    n_g = int(n * pct / 100)

    if n_g < MIN_CELLS_PER_GROUP:
        print(f"[{ct}] skip (n_per_group={n_g} < {MIN_CELLS_PER_GROUP})")
        continue

    expr = sub.obs["SLC7A5_expr"].values
    sorted_idx = np.argsort(expr)
    bottom_cells = sorted_idx[:n_g]
    top_cells    = sorted_idx[-n_g:]

    grp = np.full(n, "middle", dtype=object)
    grp[bottom_cells] = "SLC7A5_low"
    grp[top_cells]    = "SLC7A5_high"
    sub.obs["SLC7A5_pct_group"] = grp

    sub_tb = sub[sub.obs["SLC7A5_pct_group"] != "middle"].copy()

    # SLC7A5 high vs low Wilcoxon DEG
    sc.tl.rank_genes_groups(
        sub_tb,
        groupby="SLC7A5_pct_group",
        groups=["SLC7A5_high"],
        reference="SLC7A5_low",
        method="wilcoxon",
        use_raw=False,
        pts=True,
    )
    deg = sc.get.rank_genes_groups_df(sub_tb, group="SLC7A5_high").rename(columns={"names": "gene"})
    deg["logfoldchanges"] = pd.to_numeric(deg["logfoldchanges"], errors="coerce").clip(-LFC_CAP, LFC_CAP)

    ranked = (
        deg[["gene", "scores"]]
        .dropna()
        .drop_duplicates("gene")
        .sort_values("scores", ascending=False)
        .set_index("gene")["scores"]
    )
    dup_pct = ranked.duplicated().mean() * 100

    deg.to_csv(OUTDIR_ANA / f"DEG_{ct}_SLC7A5_high_vs_low.csv", index=False)

    pre_res = gp.prerank(
        rnk=ranked,
        gene_sets=GENE_SETS,
        threads=4,
        min_size=15,
        max_size=500,
        permutation_num=1000,
        outdir=None,
        seed=42,
        verbose=False,
    )
    results_pct[ct] = pre_res
    pre_res.res2d.to_csv(OUTDIR_ANA / f"GSEA_{ct}_hallmark_topbottom.csv", index=False)

    n_sig = int((pre_res.res2d["FDR q-val"].astype(float) < 0.25).sum())
    print(f"[{ct}] n/group={n_g} (top/bottom {pct}%) | dup_score={dup_pct:.1f}% | sig_pathway(FDR<0.25)={n_sig}")
    print(pre_res.res2d.sort_values("NES", ascending=False)
                       .head(5)[["Term", "NES", "NOM p-val", "FDR q-val"]].to_string(index=False))
    print()


In [ ]:
# ── 4. Hallmark GSEA NES Heatmap (Top/Bottom percentile) ────────────────────
KEYWORDS = [
    "MTORC1", "PI3K", "HIF", "AUTOPHAGY", "MYC",
    "GLYCOLYSIS", "OXIDATIVE_PHOSPHORYLATION", "FATTY_ACID",
    "INFLAMMATORY", "TNF", "INTERFERON", "IL6", "IL2",
    "G2M", "MITOTIC", "UNFOLDED", "COMPLEMENT", "COAGULATION",
    "APOPTOSIS", "P53", "ANGIOGENESIS",
]

def match_terms(df, kws):
    pattern = "|".join(kws)
    mask = df["Term"].str.upper().str.replace(" ", "_").str.replace("-", "_").str.contains(pattern, regex=True)
    return df[mask].copy()

# 유의한 경로 수집 (FDR<0.25 OR NOM p<0.05, AND keyword 매칭)
sig_terms = set()
for ct, pre_res in results_pct.items():
    df = pre_res.res2d.copy()
    df["FDR q-val"] = pd.to_numeric(df["FDR q-val"], errors="coerce")
    df["NOM p-val"] = pd.to_numeric(df["NOM p-val"], errors="coerce")
    sig = df[(df["FDR q-val"] < 0.25) | (df["NOM p-val"] < 0.05)]
    sig_terms.update(match_terms(sig, KEYWORDS)["Term"].tolist())

# 유의 경로 너무 적으면 FDR<0.25 전체 추가
if len(sig_terms) < 3:
    for ct, pre_res in results_pct.items():
        df = pre_res.res2d.copy()
        df["FDR q-val"] = pd.to_numeric(df["FDR q-val"], errors="coerce")
        sig_terms.update(df[df["FDR q-val"] < 0.25]["Term"].tolist())

# 그래도 없으면 NES 상위 10개
if len(sig_terms) < 3:
    for ct, pre_res in results_pct.items():
        sig_terms.update(pre_res.res2d.nlargest(10, "NES")["Term"].tolist())

all_terms_pct = sorted(sig_terms)
row_index_pct = list(results_pct.keys())

nes_mat  = pd.DataFrame(np.nan, index=row_index_pct, columns=all_terms_pct)
pval_mat = pd.DataFrame(np.nan, index=row_index_pct, columns=all_terms_pct)

for ct, pre_res in results_pct.items():
    df = pre_res.res2d.set_index("Term")
    for term in all_terms_pct:
        if term in df.index:
            nes_mat.loc[ct, term]  = float(df.loc[term, "NES"])
            pval_mat.loc[ct, term] = float(df.loc[term, "NOM p-val"])

nes_mat  = nes_mat.dropna(axis=1, how="all")
pval_mat = pval_mat.reindex(columns=nes_mat.columns)
all_terms_pct = nes_mat.columns.tolist()

# 시각화
fig, ax = plt.subplots(figsize=(max(len(all_terms_pct) * 0.9 + 1, 8), max(len(row_index_pct) * 0.9 + 1, 4)))

nes_vals = nes_mat.values.astype(float)
vmax = float(np.nanmax(np.abs(nes_vals))) if not np.all(np.isnan(nes_vals)) else 2.0
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cmap = plt.cm.RdBu_r

im = ax.imshow(nes_vals, cmap=cmap, norm=norm, aspect="auto")

for i, ct in enumerate(row_index_pct):
    for j, term in enumerate(all_terms_pct):
        pval = pval_mat.loc[ct, term]
        if pd.notna(pval) and float(pval) < 0.05:
            ax.text(j, i, "*", ha="center", va="center", fontsize=13, color="black", fontweight="bold")

ax.set_xticks(range(len(all_terms_pct)))
ax.set_xticklabels(all_terms_pct, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(row_index_pct)))
ax.set_yticklabels(row_index_pct, fontsize=9)
plt.colorbar(im, ax=ax, label="NES (SLC7A5 high vs low)")

pct_info = " | ".join(
    [f"{ct}: top/bottom {adaptive_pct(int((adata_sub2.obs['celltype']==ct).sum()))}%" for ct in row_index_pct]
)
ax.set_title(
    f"Hallmark GSEA: SLC7A5 Top vs Bottom percentile\n{pct_info}\n(* NOM p<0.05)",
    fontsize=9,
)
plt.tight_layout()

plt.savefig(OUTDIR_FIG / "Figure1E_Hallmark_GSEA_topbottom_pct.pdf", bbox_inches="tight", dpi=300)
plt.savefig(OUTDIR_FIG / "Figure1E_Hallmark_GSEA_topbottom_pct.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved -> Figure1E_Hallmark_GSEA_topbottom_pct.pdf / .png")
print(f"DEG & GSEA CSV -> {OUTDIR_ANA}")


In [ ]:
# ── Fig 1E 관심 pathway 선택 heatmap ──────────────────────────────────────
# Hepatocytes / Stellate / Myeloid 3개 celltype
# 관심 pathway: Glycolysis, Gluconeogenesis, TCA cycle, FAO, NFkB, OxPhos

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

FOCUS_CELLTYPES = ["Hepatocytes", "Stellete", "Myeloid"]
OUTDIR_FIG = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure")

# results_pct는 Cell 96에서 생성됨
missing = [ct for ct in FOCUS_CELLTYPES if ct not in results_pct]
if missing:
    print(f"[WARNING] 아래 celltypes는 Cell 96 결과에 없음: {missing}")
    print("Cell 96을 먼저 실행하세요.")

# ── 관심 pathway 키워드 → Hallmark 실제 term 매핑 ─────────────
# Hallmark 2020 실제 term명 기준 키워드 (대소문자 무관)
PATHWAY_KEYWORDS = {
    "Glycolysis":       ["GLYCOLYSIS"],
    "Gluconeogenesis":  ["GLUCONEOGENESIS"],
    "TCA Cycle":        ["TCA", "TRICARBOXYLIC"],
    "FAO":              ["FATTY_ACID", "BETA.OXIDATION", "FAO"],
    "NFkB":             ["NF.KB", "NFKB", "TNF"],
    "OxPhos":           ["OXIDATIVE_PHOSPHORYLATION", "OXPHOS"],
}

def find_term(res2d, kws):
    """키워드 리스트 중 하나라도 매칭되는 첫 번째 term 반환."""
    for kw in kws:
        mask = res2d["Term"].str.upper().str.replace(" ", "_").str.replace("-", "_").str.contains(kw, regex=True)
        matched = res2d[mask]
        if not matched.empty:
            # NES 절댓값 최대인 term 선택
            return matched.loc[matched["NES"].abs().idxmax(), "Term"]
    return None

# ── celltype × 관심pathway NES/pval 매트릭스 ──────────────────
# row: 관심 pathway label, col: celltype
pathway_labels = list(PATHWAY_KEYWORDS.keys())
nes_rows  = {}
pval_rows = {}
term_map  = {}   # pathway_label -> {ct: matched_term}

print(f"{'Pathway':<18} ", end="")
for ct in FOCUS_CELLTYPES:
    print(f"{ct:<20}", end="")
print()
print("-" * (18 + 20 * len(FOCUS_CELLTYPES)))

for label, kws in PATHWAY_KEYWORDS.items():
    nes_rows[label]  = {}
    pval_rows[label] = {}
    term_map[label]  = {}
    print(f"{label:<18} ", end="")
    for ct in FOCUS_CELLTYPES:
        if ct not in results_pct:
            nes_rows[label][ct]  = np.nan
            pval_rows[label][ct] = np.nan
            print(f"{'(no result)':<20}", end="")
            continue
        res2d = results_pct[ct].res2d.copy()
        res2d["NES"]       = pd.to_numeric(res2d["NES"],       errors="coerce")
        res2d["NOM p-val"] = pd.to_numeric(res2d["NOM p-val"], errors="coerce")
        matched_term = find_term(res2d, kws)
        term_map[label][ct] = matched_term
        if matched_term is not None:
            row = res2d[res2d["Term"] == matched_term].iloc[0]
            nes_rows[label][ct]  = float(row["NES"])
            pval_rows[label][ct] = float(row["NOM p-val"])
            p_str = f"(p={row['NOM p-val']:.3f})"
            print(f"{row['NES']:+.3f} {p_str:<13}", end="")
        else:
            nes_rows[label][ct]  = np.nan
            pval_rows[label][ct] = np.nan
            print(f"{'(not found)':<20}", end="")
    print()

nes_df  = pd.DataFrame(nes_rows,  index=FOCUS_CELLTYPES).T  # row=pathway, col=celltype
pval_df = pd.DataFrame(pval_rows, index=FOCUS_CELLTYPES).T

# (not found) 경로 알림
print()
for label, ct_map in term_map.items():
    not_found = [ct for ct, t in ct_map.items() if t is None]
    if not_found:
        print(f"[INFO] '{label}' — Hallmark에서 매칭된 term 없음: {not_found}")
        print(f"       -> Hallmark 2020에 해당 pathway가 없을 수 있음 (KEGG/Reactome 고려)")

# ── Heatmap 시각화 ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(len(FOCUS_CELLTYPES) * 2.0 + 1.5, len(pathway_labels) * 0.85 + 1.5))

nes_vals = nes_df.values.astype(float)
vmax = float(np.nanmax(np.abs(nes_vals))) if not np.all(np.isnan(nes_vals)) else 2.0
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cmap = plt.cm.RdBu_r

im = ax.imshow(nes_vals, cmap=cmap, norm=norm, aspect="auto")

# 유의성 표시: * NOM p<0.05, ** NOM p<0.01
for i, label in enumerate(pathway_labels):
    for j, ct in enumerate(FOCUS_CELLTYPES):
        pval = pval_df.loc[label, ct]
        nes  = nes_df.loc[label, ct]
        if pd.isna(pval) or pd.isna(nes):
            ax.text(j, i, "NA", ha="center", va="center", fontsize=7, color="gray")
        elif float(pval) < 0.01:
            ax.text(j, i, "**", ha="center", va="center", fontsize=11, color="black", fontweight="bold")
        elif float(pval) < 0.05:
            ax.text(j, i, "*",  ha="center", va="center", fontsize=12, color="black", fontweight="bold")

ax.set_xticks(range(len(FOCUS_CELLTYPES)))
ax.set_xticklabels(FOCUS_CELLTYPES, fontsize=11)
ax.set_yticks(range(len(pathway_labels)))
ax.set_yticklabels(pathway_labels, fontsize=11)
ax.set_xlabel("Cell type", fontsize=11)

plt.colorbar(im, ax=ax, label="NES (SLC7A5 high vs low)")

# percentile 정보
pct_info = " / ".join(
    [f"{ct}: top/bottom {adaptive_pct(int((adata_sub2.obs['celltype']==ct).sum()))}%"
     for ct in FOCUS_CELLTYPES if ct in results_pct]
)
ax.set_title(
    f"Hallmark GSEA: SLC7A5 High vs Low\n{pct_info}\n(** p<0.01, * p<0.05, NOM p-val)",
    fontsize=9,
)
plt.tight_layout()

plt.savefig(OUTDIR_FIG / "Figure1E_Hallmark_GSEA_focused.pdf", bbox_inches="tight", dpi=300)
plt.savefig(OUTDIR_FIG / "Figure1E_Hallmark_GSEA_focused.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved -> Figure1E_Hallmark_GSEA_focused.pdf / .png")


In [ ]:
# ── Fig 1E: SLC7A5 Top/Bottom Percentile KEGG GSEA ─────────────────────────
# Hallmark에 없는 Gluconeogenesis / TCA cycle 등을 KEGG로 확인
# adata_sub2, adaptive_pct, MIN_CELLS_PER_GROUP 은 Cell 96에서 정의됨

import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

GENE_SETS_KEGG = "KEGG_2021_Human"
FOCUS_CELLTYPES = ["Hepatocytes", "Stellete", "Myeloid"]
LFC_CAP = 10.0
MIN_CELLS_PER_GROUP = 25   # Hepatocytes 포함을 위해 25로 설정
OUTDIR_FIG = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure")
OUTDIR_ANA = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/SLC7A5_topbottom_pct")

results_kegg = {}

for ct in FOCUS_CELLTYPES:
    sub = adata_sub2[adata_sub2.obs["celltype"] == ct].copy()
    n = len(sub)
    pct = adaptive_pct(n)
    n_g = int(n * pct / 100)

    if n_g < MIN_CELLS_PER_GROUP:
        print(f"[{ct}] skip (n_per_group={n_g} < {MIN_CELLS_PER_GROUP})")
        continue

    expr = sub.obs["SLC7A5_expr"].values
    sorted_idx = np.argsort(expr)
    grp = np.full(n, "middle", dtype=object)
    grp[sorted_idx[:n_g]]  = "SLC7A5_low"
    grp[sorted_idx[-n_g:]] = "SLC7A5_high"
    sub.obs["SLC7A5_pct_group"] = grp

    sub_tb = sub[sub.obs["SLC7A5_pct_group"] != "middle"].copy()

    sc.tl.rank_genes_groups(
        sub_tb,
        groupby="SLC7A5_pct_group",
        groups=["SLC7A5_high"],
        reference="SLC7A5_low",
        method="wilcoxon",
        use_raw=False,
        pts=True,
    )
    deg = sc.get.rank_genes_groups_df(sub_tb, group="SLC7A5_high").rename(columns={"names": "gene"})
    deg["logfoldchanges"] = pd.to_numeric(deg["logfoldchanges"], errors="coerce").clip(-LFC_CAP, LFC_CAP)

    ranked = (
        deg[["gene", "scores"]]
        .dropna()
        .drop_duplicates("gene")
        .sort_values("scores", ascending=False)
        .set_index("gene")["scores"]
    )

    pre_res = gp.prerank(
        rnk=ranked,
        gene_sets=GENE_SETS_KEGG,
        threads=4,
        min_size=15,
        max_size=500,
        permutation_num=1000,
        outdir=None,
        seed=42,
        verbose=False,
    )
    results_kegg[ct] = pre_res
    pre_res.res2d.to_csv(OUTDIR_ANA / f"GSEA_{ct}_kegg_topbottom.csv", index=False)

    n_sig = int((pre_res.res2d["FDR q-val"].astype(float) < 0.25).sum())
    print(f"[{ct}] n/group={n_g} (top/bottom {pct}%) | sig_pathway(FDR<0.25)={n_sig}")
    print(pre_res.res2d.sort_values("NES", ascending=False)
                       .head(5)[["Term", "NES", "NOM p-val", "FDR q-val"]].to_string(index=False))
    print()


In [ ]:
# ── KEGG 관심 pathway 선택 heatmap ──────────────────────────────────────────
# KEGG_2021_Human 실제 term명 기준 키워드

KEGG_PATHWAY_KEYWORDS = {
    "Glycolysis /\nGluconeogenesis": ["GLYCOLYSIS", "GLUCONEOGENESIS"],
    "TCA Cycle":                      ["CITRATE_CYCLE", "TCA", "TRICARBOXYLIC"],
    "FAO":                            ["FATTY_ACID_DEGRADATION", "BETA.OXIDATION", "FATTY_ACID_METABOLISM"],
    "NFkB":                           ["NF.KAPPA_B", "NFKB", "TNF"],
    "OxPhos":                         ["OXIDATIVE_PHOSPHORYLATION"],
}

def find_term(res2d, kws):
    for kw in kws:
        mask = res2d["Term"].str.upper().str.replace(" ", "_").str.replace("-", "_").str.replace("/", "_").str.contains(kw, regex=True)
        matched = res2d[mask]
        if not matched.empty:
            return matched.loc[matched["NES"].abs().idxmax(), "Term"]
    return None

pathway_labels = list(KEGG_PATHWAY_KEYWORDS.keys())
nes_rows  = {}
pval_rows = {}
term_map  = {}

print(f"{'Pathway':<25} ", end="")
for ct in FOCUS_CELLTYPES:
    print(f"{ct:<25}", end="")
print()
print("-" * (25 + 25 * len(FOCUS_CELLTYPES)))

for label, kws in KEGG_PATHWAY_KEYWORDS.items():
    nes_rows[label]  = {}
    pval_rows[label] = {}
    term_map[label]  = {}
    display_label = label.replace("\n", " ")
    print(f"{display_label:<25} ", end="")
    for ct in FOCUS_CELLTYPES:
        if ct not in results_kegg:
            nes_rows[label][ct]  = np.nan
            pval_rows[label][ct] = np.nan
            print(f"{'(no result)':<25}", end="")
            continue
        res2d = results_kegg[ct].res2d.copy()
        res2d["NES"]       = pd.to_numeric(res2d["NES"],       errors="coerce")
        res2d["NOM p-val"] = pd.to_numeric(res2d["NOM p-val"], errors="coerce")
        matched_term = find_term(res2d, kws)
        term_map[label][ct] = matched_term
        if matched_term is not None:
            row = res2d[res2d["Term"] == matched_term].iloc[0]
            nes_rows[label][ct]  = float(row["NES"])
            pval_rows[label][ct] = float(row["NOM p-val"])
            p_str = f"(p={row['NOM p-val']:.3f})"
            print(f"{row['NES']:+.3f} {p_str:<18}", end="")
        else:
            nes_rows[label][ct]  = np.nan
            pval_rows[label][ct] = np.nan
            print(f"{'(not found)':<25}", end="")
    print()

nes_df  = pd.DataFrame(nes_rows,  index=FOCUS_CELLTYPES).T
pval_df = pd.DataFrame(pval_rows, index=FOCUS_CELLTYPES).T

# 매칭된 실제 term명 확인
print()
print("=== 매칭된 KEGG term명 ===")
for label, ct_map in term_map.items():
    display_label = label.replace("\n", " ")
    for ct, t in ct_map.items():
        status = t if t else "(not found)"
        print(f"  {display_label} | {ct}: {status}")

# ── 시각화 ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(len(FOCUS_CELLTYPES) * 2.2 + 1.5, len(pathway_labels) * 0.9 + 1.5))

nes_vals = nes_df.values.astype(float)
vmax = float(np.nanmax(np.abs(nes_vals))) if not np.all(np.isnan(nes_vals)) else 2.0
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cmap = plt.cm.RdBu_r

im = ax.imshow(nes_vals, cmap=cmap, norm=norm, aspect="auto")

for i, label in enumerate(pathway_labels):
    for j, ct in enumerate(FOCUS_CELLTYPES):
        pval = pval_df.loc[label, ct]
        nes  = nes_df.loc[label, ct]
        if pd.isna(pval) or pd.isna(nes):
            ax.text(j, i, "NA", ha="center", va="center", fontsize=7, color="gray")
        elif float(pval) < 0.01:
            ax.text(j, i, "**", ha="center", va="center", fontsize=11, color="black", fontweight="bold")
        elif float(pval) < 0.05:
            ax.text(j, i, "*",  ha="center", va="center", fontsize=12, color="black", fontweight="bold")

ax.set_xticks(range(len(FOCUS_CELLTYPES)))
ax.set_xticklabels(FOCUS_CELLTYPES, fontsize=11)
ax.set_yticks(range(len(pathway_labels)))
ax.set_yticklabels([l.replace("\n", "\n") for l in pathway_labels], fontsize=11)
ax.set_xlabel("Cell type", fontsize=11)
plt.colorbar(im, ax=ax, label="NES (SLC7A5 high vs low)")

pct_info = " / ".join(
    [f"{ct}: top/bottom {adaptive_pct(int((adata_sub2.obs['celltype']==ct).sum()))}%"
     for ct in FOCUS_CELLTYPES if ct in results_kegg]
)
ax.set_title(
    f"KEGG GSEA: SLC7A5 High vs Low\n{pct_info}\n(** p<0.01, * p<0.05, NOM p-val)",
    fontsize=9,
)
plt.tight_layout()

plt.savefig(OUTDIR_FIG / "Figure1E_KEGG_GSEA_focused.pdf", bbox_inches="tight", dpi=300)
plt.savefig(OUTDIR_FIG / "Figure1E_KEGG_GSEA_focused.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved -> Figure1E_KEGG_GSEA_focused.pdf / .png")


In [ ]:
# ── Fig 1E add-on: SLC7A5 Top/Bottom GSEA in Stellate/Myeloid only ─────────
# Hepatocytes are excluded because SLC7A5 detection is too sparse.
# Note: this object uses the celltype label "Stellete".

import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

_MONO_H5AD = "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_monoMAC_detail.h5ad"
adata_mono_sm = sc.read(_MONO_H5AD)

GENE = "SLC7A5"
FOCUS_CELLTYPES_SM = ["Stellete", "Myeloid"]
GENE_SET_COLLECTIONS_SM = {
    "hallmark": "MSigDB_Hallmark_2020",
    "kegg": "KEGG_2021_Human",
}
LFC_CAP = 10.0
MIN_CELLS_PER_GROUP = 25
OUTDIR_FIG = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure")
OUTDIR_ANA_SM = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/SLC7A5_topbottom_pct_stellate_myeloid_only")
OUTDIR_ANA_SM.mkdir(parents=True, exist_ok=True)

missing_ct = [ct for ct in FOCUS_CELLTYPES_SM if ct not in adata_mono_sm.obs["celltype"].astype(str).unique()]
if missing_ct:
    raise ValueError(f"Missing celltype labels in adata_mono_sm.obs['celltype']: {missing_ct}")

adata_sm = adata_mono_sm[adata_mono_sm.obs["celltype"].isin(FOCUS_CELLTYPES_SM)].copy()
expr_vec = adata_sm[:, GENE].X
expr_vec = expr_vec.toarray().flatten() if hasattr(expr_vec, "toarray") else np.asarray(expr_vec).flatten()
adata_sm.obs["SLC7A5_expr"] = expr_vec

def adaptive_pct(n_cells):
    if n_cells >= 1000:
        return 10
    elif n_cells >= 500:
        return 15
    else:
        return 20

def run_slc7a5_topbottom_gsea(adata_in, celltypes, gene_sets, label):
    results = {}
    summary_rows = []

    print(f"=== {label.upper()} | SLC7A5 high vs low | Stellate/Myeloid only ===")
    print(f"{'CellType':<12} {'Total':>7} {'Pct':>5} {'N/group':>8} {'Detected':>9}")
    print("-" * 50)

    for ct in celltypes:
        sub = adata_in[adata_in.obs["celltype"] == ct].copy()
        n = len(sub)
        pct = adaptive_pct(n)
        n_g = int(n * pct / 100)
        detected = int((sub.obs["SLC7A5_expr"] > 0).sum())
        print(f"{ct:<12} {n:>7} {pct:>4}% {n_g:>8} {detected:>9}")

        if n_g < MIN_CELLS_PER_GROUP:
            print(f"[{ct}] skip (n_per_group={n_g} < {MIN_CELLS_PER_GROUP})")
            continue

        expr = sub.obs["SLC7A5_expr"].to_numpy()
        sorted_idx = np.argsort(expr)
        grp = np.full(n, "middle", dtype=object)
        grp[sorted_idx[:n_g]] = "SLC7A5_low"
        grp[sorted_idx[-n_g:]] = "SLC7A5_high"
        sub.obs["SLC7A5_pct_group"] = grp

        sub_tb = sub[sub.obs["SLC7A5_pct_group"] != "middle"].copy()
        sc.tl.rank_genes_groups(
            sub_tb,
            groupby="SLC7A5_pct_group",
            groups=["SLC7A5_high"],
            reference="SLC7A5_low",
            method="wilcoxon",
            use_raw=False,
            pts=True,
        )
        deg = sc.get.rank_genes_groups_df(sub_tb, group="SLC7A5_high").rename(columns={"names": "gene"})
        deg["logfoldchanges"] = pd.to_numeric(deg["logfoldchanges"], errors="coerce").clip(-LFC_CAP, LFC_CAP)
        deg.to_csv(OUTDIR_ANA_SM / f"DEG_{ct}_SLC7A5_high_vs_low_{label}.csv", index=False)

        ranked = (
            deg[["gene", "scores"]]
            .dropna()
            .drop_duplicates("gene")
            .sort_values("scores", ascending=False)
            .set_index("gene")["scores"]
        )
        dup_pct = ranked.duplicated().mean() * 100

        pre_res = gp.prerank(
            rnk=ranked,
            gene_sets=gene_sets,
            threads=4,
            min_size=15,
            max_size=500,
            permutation_num=1000,
            outdir=None,
            seed=42,
            verbose=False,
        )
        results[ct] = pre_res
        pre_res.res2d.to_csv(OUTDIR_ANA_SM / f"GSEA_{ct}_{label}_topbottom.csv", index=False)

        n_sig = int((pre_res.res2d["FDR q-val"].astype(float) < 0.25).sum())
        summary_rows.append({
            "gene_set": label,
            "cell_type": ct,
            "total_cells": n,
            "top_bottom_pct": pct,
            "n_per_group": n_g,
            "slc7a5_detected_cells": detected,
            "duplicated_score_pct": dup_pct,
            "sig_pathway_fdr_lt_0.25": n_sig,
        })
        print(f"[{ct}] n/group={n_g} | dup_score={dup_pct:.1f}% | sig_pathway(FDR<0.25)={n_sig}")
        print(pre_res.res2d.sort_values("NES", ascending=False)
                           .head(5)[["Term", "NES", "NOM p-val", "FDR q-val"]].to_string(index=False))
        print()

    return results, pd.DataFrame(summary_rows)

results_sm = {}
summary_sm = []
for label, gene_sets in GENE_SET_COLLECTIONS_SM.items():
    res, summary = run_slc7a5_topbottom_gsea(adata_sm, FOCUS_CELLTYPES_SM, gene_sets, label)
    results_sm[label] = res
    summary_sm.append(summary)

summary_sm = pd.concat(summary_sm, ignore_index=True) if summary_sm else pd.DataFrame()
summary_sm.to_csv(OUTDIR_ANA_SM / "SLC7A5_topbottom_stellate_myeloid_only_summary.csv", index=False)
summary_sm
